# Data, Labels, CV & Sanity Guards

In [1]:
# ============================================================
# STAGE 1 — Data, Labels, CV & Sanity Guards (ONE CELL, FINAL)
# English stage title, Indonesian explanations.
#
# Struktur dataset (sesuai screenshot kamu):
# - train_images/authentic/{case_id}.png  -> label authentic (mask = empty)
# - train_images/forged/{case_id}.png     -> label forged (mask ada di train_masks/{case_id}.npy)
# - train_masks/{case_id}.npy             -> GT mask untuk forged
# - supplemental_images/{case_id}.png
# - supplemental_masks/{case_id}.npy
# - test_images/{case_id}.png
# - sample_submission.csv (case_id, annotation)
#
# Tujuan:
# - Bangun df_train_all (sample-level, UNIQUE per sample_id):
#     sample_id, case_id, variant(auth/forg/supp), image_path, mask_paths(list), n_masks, y_forged, fold
# - Bangun df_train_seg (hanya forged/supp untuk training segmentasi)
# - df_test aligned ke sample_submission order
# - Sanity:
#   * cek counts folder
#   * cek mask->image align (transpose/reshape/resize) untuk beberapa sample
#   * RLE encode/decode roundtrip (order='F')
# - CV:
#   * fold dibuat PER-GROUP case_id agar auth+forg (pasangan) selalu di fold yang sama
#   * stratify pakai has_auth (supaya distribusi negative lebih rata)
# ============================================================

import re, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold

try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# PATHS (sesuai request)
# ----------------------------
DATA_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
PATHS = {
    "supp_images":  DATA_ROOT / "supplemental_images",
    "supp_masks":   DATA_ROOT / "supplemental_masks",
    "test_images":  DATA_ROOT / "test_images",
    "train_images": DATA_ROOT / "train_images",
    "train_masks":  DATA_ROOT / "train_masks",
    "sample_sub":   DATA_ROOT / "sample_submission.csv",
}
DINO_BASE_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")

for k,v in PATHS.items():
    if not v.exists():
        raise FileNotFoundError(f"PATH tidak ditemukan: {k} -> {v}")
if not DINO_BASE_DIR.exists():
    raise FileNotFoundError(f"DINOv2 base path tidak ditemukan: {DINO_BASE_DIR}")

print("OK PATHS:")
for k,v in PATHS.items():
    print(f"  {k:>12}: {v}")
print(f"  {'dino_base':>12}: {DINO_BASE_DIR}")
print("scipy:", _HAS_SCIPY)

# ----------------------------
# Diagnostics / indexing helpers
# ----------------------------
IMG_EXTS  = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
MASK_EXTS = {".npy",".npz"} | IMG_EXTS

def folder_stats(folder: Path, allow_ext=None, show=6):
    files = [p for p in folder.rglob("*") if p.is_file()]
    if allow_ext is not None:
        files = [p for p in files if p.suffix.lower() in allow_ext]
    exts = Counter([p.suffix.lower() for p in files])
    print(f"\n[{folder}] files={len(files):,} ext_counts={dict(exts.most_common(8))}")
    for p in files[:show]:
        print("  -", p.relative_to(folder))
    return files

def list_images(folder: Path):
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    return files

def list_masks(folder: Path):
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in MASK_EXTS]
    files.sort()
    return files

def build_map_by_stem(files):
    mp = {}
    for p in files:
        mp.setdefault(p.stem, p)
    return mp

# folder stats (penting biar gak salah jalur)
_ = folder_stats(PATHS["train_images"], allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["train_masks"],  allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["supp_images"],  allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["supp_masks"],   allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["test_images"],  allow_ext=IMG_EXTS)

# ----------------------------
# Index train images by variant (auth/forg)
# ----------------------------
train_files = list_images(PATHS["train_images"])
train_auth = {}
train_forg = {}
for p in train_files:
    parts = [x.lower() for x in p.parts]
    cid = p.stem
    if "authentic" in parts:
        train_auth.setdefault(cid, p)
    elif "forged" in parts:
        train_forg.setdefault(cid, p)

supp_files = list_images(PATHS["supp_images"])
test_files = list_images(PATHS["test_images"])
supp_map = build_map_by_stem(supp_files)
test_map = build_map_by_stem(test_files)

overlap = set(train_auth) & set(train_forg)
print("\nIndexed images:")
print(f"  train/authentic: {len(train_auth):,}")
print(f"  train/forged   : {len(train_forg):,}")
print(f"  overlap ids    : {len(overlap):,} (ini NORMAL: pasangan auth+forg)")
print(f"  supplemental   : {len(supp_map):,}")
print(f"  test           : {len(test_map):,}")

# ----------------------------
# Index masks (group per case_id)
# train_masks/{id}.npy & supplemental_masks/{id}.npy
# ----------------------------
def mask_case_id_from_stem(stem: str):
    nums = re.findall(r"\d+", stem)
    if nums:
        nums = sorted(nums, key=len, reverse=True)
        return nums[0]
    return stem.split("_")[0]

def group_masks(files):
    mp = defaultdict(list)
    for p in files:
        cid = str(mask_case_id_from_stem(p.stem))
        mp[cid].append(p)
    return dict(mp)

train_mask_files = list_masks(PATHS["train_masks"])
supp_mask_files  = list_masks(PATHS["supp_masks"])
train_mask_map = group_masks(train_mask_files)
supp_mask_map  = group_masks(supp_mask_files)

print("\nGrouped masks:")
print(f"  train_masks groups: {len(train_mask_map):,} | files={len(train_mask_files):,}")
print(f"  supp_masks  groups: {len(supp_mask_map):,} | files={len(supp_mask_files):,}")

def get_mask_list(cid: str, prefer_train=True):
    # untuk train_forged: prefer train_masks
    lst = []
    if prefer_train and cid in train_mask_map:
        lst += train_mask_map[cid]
    if (not prefer_train) and cid in supp_mask_map:
        lst += supp_mask_map[cid]
    # tambahkan sisanya
    if cid in train_mask_map and (not prefer_train):
        lst += train_mask_map[cid]
    if cid in supp_mask_map and prefer_train:
        lst += supp_mask_map[cid]
    # unique
    seen = set()
    out = []
    for p in lst:
        sp = str(p)
        if sp not in seen:
            out.append(p); seen.add(sp)
    return out

# ----------------------------
# Mask loader + ALIGN to image shape
# (biar sanity mismatch=0 seperti output kamu yang terakhir)
# ----------------------------
def load_mask_any(path: str) -> np.ndarray:
    p = Path(path)
    ext = p.suffix.lower()
    if ext in IMG_EXTS:
        return np.array(Image.open(p).convert("L"))
    if ext == ".npy":
        return np.array(np.load(p))
    if ext == ".npz":
        z = np.load(p)
        for k in z.files:
            a = z[k]
            if hasattr(a, "ndim") and a.ndim in (2,3):
                return np.array(a)
        raise ValueError(f"NPZ tanpa array 2D/3D: {p}")
    raise ValueError(f"Mask ext tidak didukung: {ext} ({p})")

def to_binary_mask(arr: np.ndarray) -> np.ndarray:
    a = np.array(arr)
    if a.ndim == 3:
        # union-kan instance/channel
        if a.shape[-1] <= 4:
            a = np.max(a, axis=-1)
        else:
            a = np.max(a, axis=0)
    if np.issubdtype(a.dtype, np.floating):
        return (a > 0.5).astype(np.uint8)
    return (a > 0).astype(np.uint8)

def align_mask_to_image(mask2d: np.ndarray, H: int, W: int) -> np.ndarray:
    m = np.array(mask2d)
    if m.shape == (H, W):
        return m.astype(np.uint8)
    if m.shape == (W, H):
        return m.T.astype(np.uint8)
    if m.size == H * W:
        try:
            return m.reshape(H, W).astype(np.uint8)
        except Exception:
            pass
    # resize nearest
    im = Image.fromarray((m > 0).astype(np.uint8) * 255)
    im = im.resize((W, H), resample=Image.NEAREST)
    return (np.array(im) > 0).astype(np.uint8)

def union_masks_aligned(mask_paths: list, H: int, W: int) -> np.ndarray:
    if not mask_paths:
        return None
    out = np.zeros((H, W), dtype=np.uint8)
    for p in mask_paths:
        raw = load_mask_any(p)
        bm  = to_binary_mask(raw)
        am  = align_mask_to_image(bm, H, W)
        out = np.maximum(out, am)
    return out

# ----------------------------
# Build df_train_all (SAMPLE-LEVEL, UNIQUE per sample_id)
# - auth: y_forged=0, no mask
# - forg: y_forged=1, mask from train_masks
# - supp: y_forged=1, mask from supplemental_masks
# Catatan: walau case_id overlap, sample_id berbeda -> aman untuk cache fitur
# ----------------------------
rows = []

# train/auth
for cid, p in train_auth.items():
    rows.append({
        "sample_id": f"{cid}__auth",
        "case_id": str(cid),
        "variant": "auth",
        "image_path": str(p),
        "mask_paths": [],
        "n_masks": 0,
        "y_forged": 0,
    })

# train/forg
for cid, p in train_forg.items():
    mlist = get_mask_list(str(cid), prefer_train=True)
    rows.append({
        "sample_id": f"{cid}__forg",
        "case_id": str(cid),
        "variant": "forg",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": 1,  # forged image => positive
    })

# supplemental (anggap forged-style training)
for cid, p in supp_map.items():
    mlist = get_mask_list(str(cid), prefer_train=False)
    rows.append({
        "sample_id": f"{cid}__supp",
        "case_id": str(cid),
        "variant": "supp",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": 1 if len(mlist) > 0 else 0,
    })

df_train_all = pd.DataFrame(rows)
df_train_all["case_id"] = df_train_all["case_id"].astype(str)

# df_train_seg: hanya sample yang memang punya mask (untuk training segmentasi)
df_train_seg = df_train_all[df_train_all["n_masks"] > 0].reset_index(drop=True)

print("\nTrain build summary (sample-level):")
print("  df_train_all rows:", len(df_train_all))
print(df_train_all["variant"].value_counts())
print(df_train_all["y_forged"].value_counts().rename(index={0:"authentic",1:"forged"}))
print("\nSegmentation train rows (n_masks>0):", len(df_train_seg))

# ----------------------------
# df_test aligned to sample_submission
# ----------------------------
df_sample = pd.read_csv(PATHS["sample_sub"])
if not {"case_id","annotation"}.issubset({c.lower() for c in df_sample.columns}):
    raise ValueError(f"sample_submission.csv harus punya case_id & annotation. Found: {list(df_sample.columns)}")
col_case = [c for c in df_sample.columns if c.lower()=="case_id"][0]
df_sample = df_sample.rename(columns={col_case:"case_id"}).copy()
df_sample["case_id"] = df_sample["case_id"].astype(str)

df_test = pd.DataFrame({"case_id": sorted(test_map.keys())})
df_test["image_path"] = df_test["case_id"].map(lambda x: str(test_map.get(x, "")))
df_test = df_sample[["case_id"]].merge(df_test, on="case_id", how="left")
df_test["image_path"] = df_test["image_path"].fillna("").astype(str)

resolved = int(df_test["image_path"].map(lambda p: Path(p).exists()).sum())
print(f"\nTest indexed (aligned): {resolved:,}/{len(df_test):,} resolved")

# ----------------------------
# RLE utils (default Kaggle: order='F')
# ----------------------------
RLE_ORDER = "F"

def rle_decode(rle: str, shape_hw: tuple, order: str="F") -> np.ndarray:
    H, W = shape_hw
    s = str(rle).strip()
    if s == "" or s.lower() == "authentic":
        return np.zeros((H, W), dtype=np.uint8)
    nums = np.asarray(list(map(int, s.split())), dtype=np.int64)
    if len(nums) % 2 != 0:
        raise ValueError("Invalid RLE (odd length)")
    starts = nums[0::2] - 1
    lens   = nums[1::2]
    ends = starts + lens
    flat = np.zeros(H*W, dtype=np.uint8)
    for st, en in zip(starts, ends):
        st = max(0, int(st)); en = min(int(en), flat.size)
        if en > st:
            flat[st:en] = 1
    if order.upper() == "F":
        return flat.reshape((W, H)).T
    return flat.reshape((H, W))

def rle_encode(mask: np.ndarray, order: str="F") -> str:
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return ""
    pixels = m.T.reshape(-1) if order.upper() == "F" else m.reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# ----------------------------
# Sanity: cek mask align + RLE roundtrip di beberapa sample forged/supp
# ----------------------------
rng = random.Random(2025)
pool = df_train_seg.copy()
sample_n = min(20, len(pool))
sample_rows = pool.sample(sample_n, random_state=2025) if sample_n > 0 else pool

shape_mismatch = 0
roundtrip_ok = 0
nonempty = 0
comp_stats = []

for _, row in sample_rows.iterrows():
    ip = row["image_path"]
    mpaths = row["mask_paths"]
    if (not Path(ip).exists()) or (len(mpaths) == 0):
        continue
    W,H = Image.open(ip).convert("RGB").size
    um = union_masks_aligned(mpaths, H, W)
    if um is None:
        continue
    if um.shape != (H, W):
        shape_mismatch += 1
    if int(um.sum()) > 0:
        nonempty += 1
    rle = rle_encode(um, order=RLE_ORDER)
    um2 = rle_decode(rle, (H, W), order=RLE_ORDER)
    if np.array_equal(um, um2):
        roundtrip_ok += 1
    if _HAS_SCIPY:
        _, n = ndi.label(um.astype(bool))
        comp_stats.append(int(n))

print("\nSanity (seg samples):")
print(f"  sampled: {sample_n}")
print(f"  mask/image shape mismatch: {shape_mismatch}")
print(f"  non-empty union masks: {nonempty}/{sample_n}")
print(f"  RLE roundtrip ok: {roundtrip_ok}/{sample_n} (RLE_ORDER='{RLE_ORDER}')")
if _HAS_SCIPY and len(comp_stats):
    print(f"  components (min/med/max): {min(comp_stats)}/{int(np.median(comp_stats))}/{max(comp_stats)}")

# ----------------------------
# CV: fold per GROUP case_id (anti leakage)
# Karena banyak case_id punya 2 sample (auth+forg), kita assign fold di level case_id.
# Stratify pakai "has_auth" agar distribusi negatif (auth samples) merata.
# ----------------------------
N_FOLDS = 5
SEED = 2025

# group table
grp = df_train_all.groupby("case_id").agg(
    has_auth=("variant", lambda x: int("auth" in set(x))),
    n_samples=("sample_id", "count"),
).reset_index()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
grp["fold"] = -1
y_strat = grp["has_auth"].values.astype(int)  # ini yang bikin sebaran auth merata

for f, (_, va) in enumerate(skf.split(np.zeros(len(grp)), y_strat)):
    grp.loc[va, "fold"] = f

fold_map = dict(zip(grp["case_id"].astype(str), grp["fold"].astype(int)))
df_train_all["fold"] = df_train_all["case_id"].map(fold_map).astype(int)
df_train_seg["fold"] = df_train_seg["case_id"].map(fold_map).astype(int)

print(f"\nCV ready (grouped by case_id): n_splits={N_FOLDS}")
print("fold has_auth_rate:")
print(grp.groupby("fold")["has_auth"].mean().to_string())
print("\nfold sample y_forged rate (df_train_all):")
print(df_train_all.groupby("fold")["y_forged"].mean().to_string())

print("\nTrain_all head:")
print(df_train_all.head())
print("\nTrain_seg head:")
print(df_train_seg.head())
print("\nTest head:")
print(df_test.head())

print("\nDONE. Exported objects:")
print("- PATHS, DINO_BASE_DIR")
print("- df_train_all (sample-level), df_train_seg (mask-only)")
print("- df_test aligned to sample_submission")
print("- RLE_ORDER, rle_encode, rle_decode")
print("- load_mask_any, union_masks_aligned")


OK PATHS:
   supp_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_images
    supp_masks: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_masks
   test_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images
  train_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images
   train_masks: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_masks
    sample_sub: /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv
     dino_base: /kaggle/input/dinov2/pytorch/base/1
scipy: True

[/kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images] files=5,128 ext_counts={'.png': 5128}
  - forged/50028.png
  - forged/18054.png
  - forged/32154.png
  - forged/51742.png
  - forged/60154.png
  - forged/22739.png

[/kaggle/input/recodai-luc-scientific-image-forgery-detection/train_masks] files=2,751 ext_counts={'.npy': 2751}
  - 59069.n

# DINOv2-Base Feature Cache (CPU-Optimized)

In [2]:
# ============================================================
# STAGE 2 — DINOv2-Base Feature Cache (CPU-Optimized) (ONE CELL, REVISI FULL v2: Multi-Scale + Fuse + Light Whitening)
# English stage title, Indonesian explanations.
#
# Upgrade utama:
# - Multi-scale extraction (BASE + HI)
# - Resample HI grid -> BASE grid, lalu fuse (concat) jadi patch_desc lebih diskriminatif
# - Light per-image whitening (standardize per-channel across patches) untuk reduce repetitif texture dominance
# - Tetap kompatibel dengan STAGE 3: output tetap punya key "patch_desc"
#
# Output per item:
# - patch_desc: (H_p, W_p, D_fused) float16 (L2-normalized)
# - cls_desc  : (D_fused,) float16 (L2-normalized)
# - meta (JSON)
# ============================================================

import os, json, math, gc, time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn.functional as F
from transformers import AutoModel

# ----------------------------
# Guards
# ----------------------------
need_cols = {"sample_id","case_id","image_path","variant","y_forged","fold"}
if "df_train_all" not in globals() or not isinstance(df_train_all, pd.DataFrame):
    raise RuntimeError("df_train_all belum ada. Jalankan STAGE 1 dulu.")
if not need_cols.issubset(set(df_train_all.columns)):
    raise RuntimeError(f"df_train_all missing cols {need_cols - set(df_train_all.columns)}")

if "df_test" not in globals() or not isinstance(df_test, pd.DataFrame):
    raise RuntimeError("df_test belum ada. Jalankan STAGE 1 dulu.")
if not {"case_id","image_path"}.issubset(set(df_test.columns)):
    raise RuntimeError("df_test harus punya kolom case_id, image_path.")

if "DINO_BASE_DIR" not in globals():
    raise RuntimeError("DINO_BASE_DIR belum ada. Pastikan STAGE 1 mendefinisikannya.")
DINO_BASE_DIR = Path(str(DINO_BASE_DIR))
if not DINO_BASE_DIR.exists():
    raise FileNotFoundError(f"DINO_BASE_DIR tidak ditemukan: {DINO_BASE_DIR}")

# ----------------------------
# Config (CPU-friendly)
# ----------------------------
CACHE_ROOT  = Path("/kaggle/working/recodai_luc/cache/dino_v2_base")
CACHE_TRAIN = CACHE_ROOT / "train_all"
CACHE_TEST  = CACHE_ROOT / "test"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_TRAIN.mkdir(parents=True, exist_ok=True)
CACHE_TEST.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = 14

# Multi-scale:
USE_MULTI_SCALE = True
MAX_SIDE_BASE   = 384   # base grid (lebih cepat)
MAX_SIDE_HI     = 512   # hi grid (lebih detail) -> resample ke base grid
MIN_SIDE        = 224

# Fuse mode: "concat" recommended (lebih informatif, tapi D jadi 2x)
FUSE_MODE = "concat"  # "concat" atau "avg"

# Light whitening untuk reduce repetitif textures:
USE_LIGHT_WHITEN = True    # standardize per-channel across patches (per image)
WHITEN_EPS       = 1e-6

USE_FP16_STORE = True
N_LIMIT_TRAIN  = None
N_LIMIT_TEST   = None

# threads (CPU)
try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
    torch.set_num_interop_threads(1)
except Exception:
    pass

device = torch.device("cpu")

IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def _round_to_multiple(x, m):
    return int(max(m, (x // m) * m))

def resize_keep_ar(img: Image.Image, max_side: int, min_side: int, patch: int):
    """
    Resize menjaga aspect ratio.
    Long side -> clamp [min_side, max_side], lalu snap H,W ke multiple patch.
    """
    W0, H0 = img.size
    long0 = max(W0, H0)
    if long0 <= 0:
        return img, (H0, W0), (H0, W0)

    target_long = min(max_side, max(min_side, long0))
    scale = target_long / float(long0)

    W1 = max(patch, int(round(W0 * scale)))
    H1 = max(patch, int(round(H0 * scale)))

    W1 = _round_to_multiple(W1, patch)
    H1 = _round_to_multiple(H1, patch)

    if (W1, H1) != (W0, H0):
        img = img.resize((W1, H1), resample=Image.BICUBIC)

    return img, (H0, W0), (H1, W1)

def pil_to_tensor_norm(img: Image.Image):
    arr = np.array(img.convert("RGB"), dtype=np.float32) / 255.0
    arr = (arr - IMNET_MEAN) / IMNET_STD
    t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).contiguous()
    return t

def l2_normalize(x: torch.Tensor, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True) + eps)

def _infer_grid_from_tokens(N: int, H1: int, W1: int, patch: int):
    """
    Ideal grid: H1//patch, W1//patch. Jika token mismatch, coba tebak.
    """
    gh = max(1, H1 // patch)
    gw = max(1, W1 // patch)
    if gh * gw == N:
        return gh, gw

    # fallback: factorization mendekati aspect ratio
    ar = W1 / max(1, H1)
    gw0 = int(round(math.sqrt(N * ar)))
    gw0 = max(1, min(gw0, N))
    gh0 = max(1, N // gw0)
    gw0 = max(1, N // gh0)
    if gh0 * gw0 != N:
        # last resort: set gw=N, gh=1
        return 1, N
    return gh0, gw0

@torch.inference_mode()
def _forward_dino(img_pil: Image.Image):
    x = pil_to_tensor_norm(img_pil).to(device, dtype=torch.float32)
    y = model(pixel_values=x).last_hidden_state  # [1, 1+N, D]
    cls = y[:, 0, :]    # [1,D]
    pt  = y[:, 1:, :]   # [1,N,D]
    return cls, pt

def _tokens_to_grid(pt: torch.Tensor, H1: int, W1: int, patch: int):
    """
    pt: [1,N,D] -> grid [H_p,W_p,D]
    """
    N = int(pt.shape[1])
    D = int(pt.shape[2])
    gh, gw = _infer_grid_from_tokens(N, H1, W1, patch)
    pt0 = pt.squeeze(0)  # [N,D]
    pt0 = l2_normalize(pt0, dim=-1)
    # reshape
    try:
        grid = pt0.reshape(gh, gw, D).contiguous()
    except Exception:
        grid = pt0.reshape(1, N, D).contiguous()  # fallback
        gh, gw = 1, N
    return grid, gh, gw, D

def _resample_grid_to(grid: torch.Tensor, gh_t: int, gw_t: int):
    """
    grid: [gh,gw,D] -> [gh_t,gw_t,D] via bilinear on spatial dims.
    """
    gh, gw, D = grid.shape
    x = grid.permute(2,0,1).unsqueeze(0).contiguous()  # [1,D,gh,gw]
    x2 = F.interpolate(x, size=(gh_t, gw_t), mode="bilinear", align_corners=False)
    g2 = x2.squeeze(0).permute(1,2,0).contiguous()     # [gh_t,gw_t,D]
    return g2

def _light_whiten_patches(grid: torch.Tensor, eps: float = 1e-6):
    """
    Standardize per-channel across patches (per image):
    grid: [gh,gw,D] -> flatten patches -> (x-mean)/std -> L2 normalize
    """
    gh, gw, D = grid.shape
    x = grid.reshape(-1, D)
    mu = x.mean(dim=0, keepdim=True)
    sd = x.std(dim=0, keepdim=True).clamp_min(eps)
    xw = (x - mu) / sd
    xw = l2_normalize(xw, dim=-1)
    return xw.reshape(gh, gw, D).contiguous()

# ----------------------------
# Load DINOv2 Base (local)
# ----------------------------
print("\nLoading DINOv2-Base from:", DINO_BASE_DIR)
model = AutoModel.from_pretrained(str(DINO_BASE_DIR), local_files_only=True)
model.eval().to(device)

with torch.inference_mode():
    dummy = torch.zeros((1,3,224,224), dtype=torch.float32, device=device)
    out = model(pixel_values=dummy)
    D0 = int(out.last_hidden_state.shape[-1])
print("Model loaded. Base embed dim:", D0, "| device:", device)

def extract_features_one(image_path: str):
    """
    Return:
    - patch_desc: (H_p, W_p, D_fused) float32 (L2 norm)
    - cls_desc  : (D_fused,) float32 (L2 norm)
    - meta dict
    """
    img0 = Image.open(image_path).convert("RGB")

    # BASE
    img_b, (H0,W0), (Hb,Wb) = resize_keep_ar(img0, max_side=MAX_SIDE_BASE, min_side=MIN_SIDE, patch=PATCH_SIZE)
    cls_b, pt_b = _forward_dino(img_b)  # cls:[1,D0], pt:[1,N,D0]
    cls_b = l2_normalize(cls_b, dim=-1).squeeze(0)     # [D0]
    grid_b, gh, gw, D = _tokens_to_grid(pt_b, Hb, Wb, PATCH_SIZE)  # [gh,gw,D0]

    if USE_MULTI_SCALE:
        img_h, _, (Hh,Wh) = resize_keep_ar(img0, max_side=MAX_SIDE_HI, min_side=MIN_SIDE, patch=PATCH_SIZE)
        cls_h, pt_h = _forward_dino(img_h)
        cls_h = l2_normalize(cls_h, dim=-1).squeeze(0)  # [D0]
        grid_h, gh2, gw2, _ = _tokens_to_grid(pt_h, Hh, Wh, PATCH_SIZE)

        # resample HI grid -> BASE grid
        grid_h_rs = _resample_grid_to(grid_h, gh, gw)  # [gh,gw,D0]
        grid_h_rs = l2_normalize(grid_h_rs.reshape(-1, D), dim=-1).reshape(gh, gw, D).contiguous()

        if FUSE_MODE == "avg":
            grid_f = l2_normalize((grid_b + grid_h_rs), dim=-1)
            cls_f  = l2_normalize((cls_b + cls_h), dim=-1)
            D_fused = D
        else:
            grid_f = torch.cat([grid_b, grid_h_rs], dim=-1)  # [gh,gw,2D]
            grid_f = l2_normalize(grid_f.reshape(-1, grid_f.shape[-1]), dim=-1).reshape(gh, gw, -1).contiguous()
            cls_f  = torch.cat([cls_b, cls_h], dim=-1)
            cls_f  = l2_normalize(cls_f, dim=-1)
            D_fused = int(grid_f.shape[-1])
    else:
        grid_f = grid_b
        cls_f  = cls_b
        D_fused = D

    # Light whitening (reduce repetitif textures)
    if USE_LIGHT_WHITEN:
        grid_f = _light_whiten_patches(grid_f, eps=WHITEN_EPS)

    meta = {
        "orig_hw": [int(H0), int(W0)],
        "resized_hw_base": [int(Hb), int(Wb)],
        "patch_size": int(PATCH_SIZE),
        "grid_hw": [int(gh), int(gw)],
        "embed_dim_base": int(D0),
        "embed_dim_fused": int(D_fused),
        "max_side_base": int(MAX_SIDE_BASE),
        "max_side_hi": int(MAX_SIDE_HI) if USE_MULTI_SCALE else int(MAX_SIDE_BASE),
        "min_side": int(MIN_SIDE),
        "use_multi_scale": bool(USE_MULTI_SCALE),
        "fuse_mode": str(FUSE_MODE),
        "use_light_whiten": bool(USE_LIGHT_WHITEN),
    }
    return grid_f, cls_f, meta

def save_npz(path: Path, patch_desc: torch.Tensor, cls_desc: torch.Tensor, meta: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    pd_np = patch_desc.cpu().numpy()
    cd_np = cls_desc.cpu().numpy()

    if USE_FP16_STORE:
        pd_np = pd_np.astype(np.float16)
        cd_np = cd_np.astype(np.float16)
    else:
        pd_np = pd_np.astype(np.float32)
        cd_np = cd_np.astype(np.float32)

    np.savez(
        str(path),
        patch_desc=pd_np,
        cls_desc=cd_np,
        meta=json.dumps(meta),
    )

def cache_loop(df: pd.DataFrame, id_col: str, out_dir: Path, limit=None, label="train"):
    n = len(df) if limit is None else min(len(df), int(limit))
    miss = 0
    done = 0
    t0 = time.time()

    for i in range(n):
        row = df.iloc[i]
        uid = str(row[id_col])
        ip  = str(row["image_path"])
        out = out_dir / f"{uid}.npz"

        if out.exists():
            done += 1
            continue
        if (not ip) or (not Path(ip).exists()):
            miss += 1
            continue

        try:
            pt, cls, meta = extract_features_one(ip)
            save_npz(out, pt, cls, meta)
            done += 1
        except Exception as e:
            print(f"[{label}] FAIL uid={uid} path={ip} err={type(e).__name__}: {e}")
            miss += 1

        if (i+1) % 50 == 0:
            dt = time.time() - t0
            rate = (i+1)/max(dt, 1e-9)
            print(f"[{label}] {i+1}/{n} | cached={done} | miss/fail={miss} | {rate:.2f} it/s | elapsed={dt:.1f}s")
            gc.collect()

    dt = time.time() - t0
    print(f"\n[{label}] DONE. total={n} cached={done} miss/fail={miss} elapsed={dt:.1f}s")
    return {"label": label, "total": int(n), "cached": int(done), "miss_fail": int(miss), "elapsed_s": float(dt)}

# ----------------------------
# Run cache
# ----------------------------
df_train_run = df_train_all.reset_index(drop=True).copy()
df_test_run  = df_test.reset_index(drop=True).copy()

df_train_run = df_train_run.sort_values(["variant","fold","case_id","sample_id"]).reset_index(drop=True)
df_test_run  = df_test_run.sort_values(["case_id"]).reset_index(drop=True)

print("\nCaching TRAIN_ALL features...")
rep_train = cache_loop(df_train_run, id_col="sample_id", out_dir=CACHE_TRAIN, limit=N_LIMIT_TRAIN, label="train_all")

print("\nCaching TEST features...")
rep_test = cache_loop(df_test_run, id_col="case_id", out_dir=CACHE_TEST, limit=N_LIMIT_TEST, label="test")

# ----------------------------
# Write manifests
# ----------------------------
def build_manifest(df: pd.DataFrame, id_col: str, out_dir: Path):
    recs = []
    for _, r in df.iterrows():
        uid = str(r[id_col])
        p = out_dir / f"{uid}.npz"
        recs.append({
            id_col: uid,
            "case_id": str(r.get("case_id","")),
            "variant": str(r.get("variant","")),
            "fold": int(r.get("fold",-1)) if "fold" in r else -1,
            "y_forged": int(r.get("y_forged",-1)) if "y_forged" in r else -1,
            "image_path": str(r.get("image_path","")),
            "feat_path": str(p),
            "feat_exists": int(p.exists()),
        })
    return pd.DataFrame(recs)

man_train = build_manifest(df_train_run, "sample_id", CACHE_TRAIN)
man_test  = build_manifest(df_test_run,  "case_id",  CACHE_TEST)

MAN_TRAIN_PATH = CACHE_ROOT / "manifest_train_all.csv"
MAN_TEST_PATH  = CACHE_ROOT / "manifest_test.csv"
man_train.to_csv(MAN_TRAIN_PATH, index=False)
man_test.to_csv(MAN_TEST_PATH, index=False)

summary = {
    "cache_root": str(CACHE_ROOT),
    "train": rep_train,
    "test": rep_test,
    "patch_size": int(PATCH_SIZE),
    "max_side_base": int(MAX_SIDE_BASE),
    "max_side_hi": int(MAX_SIDE_HI) if USE_MULTI_SCALE else int(MAX_SIDE_BASE),
    "min_side": int(MIN_SIDE),
    "use_multi_scale": bool(USE_MULTI_SCALE),
    "fuse_mode": str(FUSE_MODE),
    "use_light_whiten": bool(USE_LIGHT_WHITEN),
    "use_fp16_store": bool(USE_FP16_STORE),
}
with open(CACHE_ROOT / "cache_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote manifests:")
print(" -", MAN_TRAIN_PATH)
print(" -", MAN_TEST_PATH)
print(" -", CACHE_ROOT / "cache_summary.json")

print("\nDONE. Exported objects:")
print("- model (DINOv2-base), CACHE_ROOT, MAN_TRAIN_PATH, MAN_TEST_PATH")
print("- helper: extract_features_one()")



Loading DINOv2-Base from: /kaggle/input/dinov2/pytorch/base/1


2025-12-31 20:00:26.058143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767211226.420036      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767211226.521539      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767211227.387310      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767211227.387364      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767211227.387366      17 computation_placer.cc:177] computation placer alr

Model loaded. Base embed dim: 768 | device: cpu

Caching TRAIN_ALL features...
[train_all] 50/5176 | cached=50 | miss/fail=0 | 0.51 it/s | elapsed=97.2s
[train_all] 100/5176 | cached=100 | miss/fail=0 | 0.50 it/s | elapsed=198.4s
[train_all] 150/5176 | cached=150 | miss/fail=0 | 0.51 it/s | elapsed=294.0s
[train_all] 200/5176 | cached=200 | miss/fail=0 | 0.51 it/s | elapsed=390.3s
[train_all] 250/5176 | cached=250 | miss/fail=0 | 0.52 it/s | elapsed=482.3s
[train_all] 300/5176 | cached=300 | miss/fail=0 | 0.51 it/s | elapsed=582.9s
[train_all] 350/5176 | cached=350 | miss/fail=0 | 0.52 it/s | elapsed=671.7s
[train_all] 400/5176 | cached=400 | miss/fail=0 | 0.52 it/s | elapsed=768.9s
[train_all] 450/5176 | cached=450 | miss/fail=0 | 0.51 it/s | elapsed=877.7s
[train_all] 500/5176 | cached=500 | miss/fail=0 | 0.51 it/s | elapsed=977.8s
[train_all] 550/5176 | cached=550 | miss/fail=0 | 0.51 it/s | elapsed=1078.0s
[train_all] 600/5176 | cached=600 | miss/fail=0 | 0.51 it/s | elapsed=1169.8

# Robust Matching (Top-k + MNN + Multi-Peak Translation)

In [3]:
# ============================================================
# STAGE 3 — Robust Matching (Top-k + TRUE MNN + Ratio/Margin + Peak NMS) (ONE CELL, REVISI FULL v4)
# English stage title, Indonesian explanations.
#
# Upgrade utama:
# - TRUE mutual nearest neighbor (MNN proper, bukan sekadar "pair muncul dua arah")
# - Ratio + margin test (mengurangi false match dari texture repetitif)
# - Valid-candidate topk: enforce far-neighbor + sim_thr sebelum ratio
# - Voting peak pakai weight = sim^wpow * margin^mpow
# - Peak NMS: peaks harus berjauhan (mengurangi offset ambigu)
#
# Output kompatibel STAGE 4:
# - best_src, best_dst, best_sim + scalars: has_peak, peak_ratio, best_weight, best_count, best_mean_sim
# ============================================================

import os, gc, time, json
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import torch

# ----------------------------
# REQUIRE (sesuai STAGE 1)
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu (df_train_all & df_test).")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

# normalisasi dtype
for c in ["sample_id", "case_id"]:
    if c in df_train_all.columns:
        df_train_all[c] = df_train_all[c].astype(str)
if "case_id" in df_test.columns:
    df_test["case_id"] = df_test["case_id"].astype(str)
if "image_path" not in df_test.columns:
    df_test["image_path"] = ""

# ----------------------------
# Locate manifests from STAGE 2
# ----------------------------
if "MAN_TRAIN_PATH" in globals():
    MAN_TRAIN_PATH = Path(str(MAN_TRAIN_PATH))
else:
    MAN_TRAIN_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv")

if "MAN_TEST_PATH" in globals():
    MAN_TEST_PATH = Path(str(MAN_TEST_PATH))
else:
    MAN_TEST_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv")

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train tidak ditemukan: {MAN_TRAIN_PATH} (jalankan STAGE 2 dulu).")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test tidak ditemukan: {MAN_TEST_PATH} (jalankan STAGE 2 dulu).")

man_train = pd.read_csv(MAN_TRAIN_PATH, dtype={"sample_id": str, "feat_path": str}, low_memory=False)
man_test  = pd.read_csv(MAN_TEST_PATH,  dtype={"case_id": str,  "feat_path": str}, low_memory=False)

need_train_cols = {"sample_id","feat_path","feat_exists"}
need_test_cols  = {"case_id","feat_path","feat_exists"}
if not need_train_cols.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all.csv missing cols: {need_train_cols - set(man_train.columns)}")
if not need_test_cols.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test.csv missing cols: {need_test_cols - set(man_test.columns)}")

man_train["sample_id"] = man_train["sample_id"].astype(str)
man_test["case_id"]    = man_test["case_id"].astype(str)
man_train["feat_path"] = man_train["feat_path"].astype(str)
man_test["feat_path"]  = man_test["feat_path"].astype(str)

man_train = man_train[man_train["feat_exists"] == 1].copy()
man_test  = man_test[man_test["feat_exists"]  == 1].copy()

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows(feat_exists=1):", len(man_train))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows(feat_exists=1):", len(man_test))

# ----------------------------
# Output dirs (pakai v3 supaya tidak ketiban)
# ----------------------------
MATCH_ROOT = Path("/kaggle/working/recodai_luc/cache/match_base_v3")
MATCH_TRAIN_DIR = MATCH_ROOT / "train_all"
MATCH_TEST_DIR  = MATCH_ROOT / "test"
MATCH_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
MATCH_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MATCH_ROOT     :", MATCH_ROOT)
print("MATCH_TRAIN_DIR:", MATCH_TRAIN_DIR)
print("MATCH_TEST_DIR :", MATCH_TEST_DIR)

# ----------------------------
# CONFIG (tuning)
# ----------------------------
CFG_MATCH = {
    # retrieval
    "topk": 80,                 # lebih besar -> kandidat lebih banyak (ratio akan menyaring)
    "sim_thr": 0.78,            # minimum cosine sim untuk dianggap kandidat
    # neighbor-close
    "min_sep": 4,               # larang match yang terlalu dekat (menghindari trivial neighbors)
    "close_metric": "max",      # "max" => max(|dr|,|dc|) < min_sep dianggap dekat
    # distinctiveness (anti-texture)
    "ratio_thr": 1.05,          # chosen_sim / second_best_sim >= ratio_thr
    "margin_thr": 0.02,         # chosen_sim - second_best_sim >= margin_thr
    # peak voting
    "bin_step": 1,
    "peaks_M": 3,
    "min_peak_count": 20,       # minimal jumlah pasangan yang voting ke peak terbaik
    "peak_min_sep": 2,          # NMS: peaks harus beda minimal ini (L_inf)
    "weight_power": 2.0,        # sim^wpow
    "margin_power": 1.0,        # margin^mpow
    # io
    "skip_if_exists": True,
    "print_every": 200,
    "limit_train": None,
    "limit_test": None,
}

try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
except Exception:
    pass

# ----------------------------
# Helpers: load DINO cache (STAGE 2 format)
# ----------------------------
def load_dino_patch_from_npz(npz_path: str):
    p = Path(npz_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)

    if "patch_desc" in z.files:
        pdsc = z["patch_desc"]
        if pdsc.ndim != 3:
            return None
        gh, gw, D = pdsc.shape
        feat = pdsc.reshape(gh*gw, D).astype(np.float32, copy=False)
        return feat, int(gh), int(gw)

    if "feat" in z.files and "grid_h" in z.files and "grid_w" in z.files:
        feat = z["feat"].astype(np.float32, copy=False)
        gh = int(z["grid_h"]); gw = int(z["grid_w"])
        return feat, gh, gw

    return None

def save_match_npz(out_path: Path, payload: dict):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(out_path), **payload)

@lru_cache(maxsize=256)
def _grid_rc_np(gh: int, gw: int):
    rr = np.repeat(np.arange(gh, dtype=np.int32), gw)
    cc = np.tile(np.arange(gw, dtype=np.int32), gh)
    return rr, cc

def _is_close(src_idx: np.ndarray, dst_idx: np.ndarray, gh: int, gw: int, min_sep: int, metric: str):
    rr, cc = _grid_rc_np(gh, gw)
    dr = np.abs(rr[src_idx] - rr[dst_idx])
    dc = np.abs(cc[src_idx] - cc[dst_idx])
    if metric == "both":
        return (dr < min_sep) & (dc < min_sep)
    # default "max"
    return (np.maximum(dr, dc) < min_sep)

def canonicalize_offsets(dy: np.ndarray, dx: np.ndarray):
    dy = dy.astype(np.int32, copy=False)
    dx = dx.astype(np.int32, copy=False)
    neg = (dy < 0) | ((dy == 0) & (dx < 0))
    dy2 = dy.copy(); dx2 = dx.copy()
    dy2[neg] = -dy2[neg]
    dx2[neg] = -dx2[neg]
    return dy2, dx2

def _encode_key(dy: np.ndarray, dx: np.ndarray):
    # key = (dy << 32) ^ (dx & 0xffffffff)
    return (dy.astype(np.int64) << 32) ^ (dx.astype(np.int64) & np.int64(0xffffffff))

def robust_match_one(feat_np: np.ndarray, gh: int, gw: int, cfg: dict):
    N = int(gh * gw)
    if feat_np is None or feat_np.ndim != 2 or feat_np.shape[0] != N:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(0),
            "n_pairs_mnn": np.int32(0),
        }

    topk = int(cfg["topk"])
    sim_thr = float(cfg["sim_thr"])
    min_sep = int(cfg["min_sep"])
    metric  = str(cfg.get("close_metric", "max"))
    ratio_thr  = float(cfg["ratio_thr"])
    margin_thr = float(cfg["margin_thr"])

    bin_step = int(cfg["bin_step"])
    peaks_M = int(cfg["peaks_M"])
    min_peak_count = int(cfg["min_peak_count"])
    peak_min_sep = int(cfg["peak_min_sep"])
    wpow = float(cfg["weight_power"])
    mpow = float(cfg["margin_power"])

    # normalize features
    f = torch.from_numpy(feat_np).to(torch.float32)
    f = torch.nn.functional.normalize(f, dim=1)

    # sim matrix (N,N)
    sim = f @ f.T
    sim.fill_diagonal_(-1e9)

    k = min(max(2, topk), N-1)
    vals, idxs = torch.topk(sim, k=k, dim=1, largest=True, sorted=True)  # sorted desc
    # candidate validity: far + sim>=thr
    # build src index grid
    src_idx = torch.arange(N, dtype=torch.int64).unsqueeze(1).expand(N, k)
    src_np = src_idx.reshape(-1).cpu().numpy().astype(np.int32, copy=False)
    dst_np = idxs.reshape(-1).cpu().numpy().astype(np.int32, copy=False)

    close_np = _is_close(src_np, dst_np, gh, gw, min_sep=min_sep, metric=metric)
    close = torch.from_numpy(close_np.reshape(N, k))

    valid = (~close) & (vals >= sim_thr)
    has_any = valid.any(dim=1)

    # masked vals for selecting best among valid
    vals_valid = vals.masked_fill(~valid, -1e9)

    best_j = torch.argmax(vals_valid, dim=1)  # index in [0,k)
    chosen_sim = vals_valid.gather(1, best_j.unsqueeze(1)).squeeze(1)     # [N]
    chosen_dst = idxs.gather(1, best_j.unsqueeze(1)).squeeze(1)          # [N]

    # second-best among valid (for ratio/margin)
    # if k==2 still ok
    top2v = torch.topk(vals_valid, k=min(2, k), dim=1, largest=True, sorted=True).values
    v1 = top2v[:, 0]
    v2 = top2v[:, 1] if top2v.shape[1] > 1 else torch.full_like(v1, -1e9)

    # ratio/margin computed on valid candidates
    eps = 1e-9
    ratio = v1 / (v2 + eps)
    margin = v1 - v2

    keep = has_any & (v1 > -1e8) & (ratio >= ratio_thr) & (margin >= margin_thr)

    if keep.sum().item() == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(int((vals >= sim_thr).sum().item())),
            "n_pairs_mnn": np.int32(0),
        }

    # build 1NN pairs after distinctiveness filter
    src_keep = torch.nonzero(keep, as_tuple=False).squeeze(1).cpu().numpy().astype(np.int32)
    dst_keep = chosen_dst[keep].cpu().numpy().astype(np.int32)
    sim_keep = chosen_sim[keep].cpu().numpy().astype(np.float32)
    mar_keep = margin[keep].cpu().numpy().astype(np.float32)

    # TRUE MNN (mutual best):
    # dst_best_src[dst] = src with max sim
    order = np.argsort(-sim_keep)  # desc sim
    dst_best_src = np.full((N,), -1, dtype=np.int32)
    dst_best_sim = np.full((N,), -1e9, dtype=np.float32)
    for idx in order:
        d = dst_keep[idx]
        if dst_best_src[d] == -1:
            dst_best_src[d] = src_keep[idx]
            dst_best_sim[d] = sim_keep[idx]

    mnn_mask = (dst_best_src[dst_keep] == src_keep)
    src_mnn = src_keep[mnn_mask]
    dst_mnn = dst_keep[mnn_mask]
    sim_mnn = sim_keep[mnn_mask]
    mar_mnn = mar_keep[mnn_mask]

    n_pairs_thr = int((vals >= sim_thr).sum().item())
    n_pairs_mnn = int(len(src_mnn))

    if n_pairs_mnn == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(0),
        }

    # offsets
    src_r = src_mnn // gw; src_c = src_mnn % gw
    dst_r = dst_mnn // gw; dst_c = dst_mnn % gw
    dy = (dst_r - src_r).astype(np.int32)
    dx = (dst_c - src_c).astype(np.int32)
    dyc, dxc = canonicalize_offsets(dy, dx)

    if bin_step > 1:
        dyb = (np.round(dyc / bin_step)).astype(np.int32) * bin_step
        dxb = (np.round(dxc / bin_step)).astype(np.int32) * bin_step
    else:
        dyb, dxb = dyc, dxc

    # vote weights
    w = np.clip(sim_mnn, 0.0, 1.0)
    if wpow != 1.0:
        w = np.power(w, wpow).astype(np.float32, copy=False)

    m = np.clip(mar_mnn, 0.0, None)
    if mpow != 1.0:
        m = np.power(m, mpow).astype(np.float32, copy=False)

    wv = (w * (m + 1e-9)).astype(np.float32, copy=False)

    keys = _encode_key(dyb, dxb)
    uniq, inv = np.unique(keys, return_inverse=True)
    sum_w = np.bincount(inv, weights=wv, minlength=len(uniq)).astype(np.float32)
    cnt   = np.bincount(inv, minlength=len(uniq)).astype(np.int32)

    # sort candidate peaks by sum_w
    order_pk = np.argsort(-sum_w)

    # decode helper
    def _decode_key(key64: int):
        dy_pk = np.int32(key64 >> 32)
        dx_pk = np.int32(key64 & np.int64(0xffffffff))
        if dx_pk >= 2**31:
            dx_pk = dx_pk - 2**32
        return int(dy_pk), int(dx_pk)

    # Peak NMS: peaks must be separated
    peaks = []
    for idx in order_pk:
        if cnt[idx] < min_peak_count:
            continue
        dy_pk, dx_pk = _decode_key(int(uniq[idx]))
        ok = True
        for (pdy, pdx, _, _) in peaks:
            if max(abs(dy_pk - pdy), abs(dx_pk - pdx)) < peak_min_sep:
                ok = False
                break
        if not ok:
            continue
        peaks.append((dy_pk, dx_pk, float(sum_w[idx]), int(cnt[idx])))
        if len(peaks) >= peaks_M:
            break

    if len(peaks) == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(n_pairs_mnn),
        }

    # peak ratio (best vs second among selected peaks)
    w1 = peaks[0][2]
    w2 = peaks[1][2] if len(peaks) > 1 else 0.0
    peak_ratio = float(w1 / (w2 + 1e-9)) if w2 > 0 else float(1e9)

    best_dy, best_dx, best_weight, best_count = peaks[0]
    best_key = _encode_key(np.array([best_dy], dtype=np.int32), np.array([best_dx], dtype=np.int32))[0]
    same = (keys == best_key)

    best_src = src_mnn[same].astype(np.int32, copy=False)
    best_dst = dst_mnn[same].astype(np.int32, copy=False)
    best_sim = sim_mnn[same].astype(np.float16, copy=False)
    best_mean_sim = float(np.mean(sim_mnn[same])) if best_src.size > 0 else 0.0

    peaks_arr   = np.array([[p[0], p[1]] for p in peaks], dtype=np.int16)
    weights_arr = np.array([p[2] for p in peaks], dtype=np.float32)
    counts_arr  = np.array([p[3] for p in peaks], dtype=np.int32)

    return {
        "grid_h": np.int32(gh),
        "grid_w": np.int32(gw),
        "has_peak": np.int8(1),
        "peak_ratio": np.float32(peak_ratio),
        "peaks_dy_dx": peaks_arr,
        "peaks_weight": weights_arr,
        "peaks_count": counts_arr,
        "best_src": best_src,
        "best_dst": best_dst,
        "best_sim": best_sim,
        "best_weight": np.float32(best_weight),
        "best_count": np.int32(best_count),
        "best_mean_sim": np.float32(best_mean_sim),
        "n_pairs_thr": np.int32(n_pairs_thr),
        "n_pairs_mnn": np.int32(n_pairs_mnn),
    }

# ----------------------------
# Build processing lists (drop overlap columns BEFORE join)
# ----------------------------
train_meta_cols = [c for c in ["sample_id","case_id","variant","fold","y_forged","image_path"] if c in df_train_all.columns]
train_meta = df_train_all[train_meta_cols].copy()
train_meta["sample_id"] = train_meta["sample_id"].astype(str)
train_meta = train_meta.drop_duplicates(subset=["sample_id"], keep="first").set_index("sample_id")

train_list = man_train.copy()
train_list["sample_id"] = train_list["sample_id"].astype(str)
overlap_cols = [c for c in train_meta.columns if c in train_list.columns]
if overlap_cols:
    train_list = train_list.drop(columns=overlap_cols)
train_list = train_list.join(train_meta, on="sample_id")

test_meta = df_test[["case_id","image_path"]].copy()
test_meta["case_id"] = test_meta["case_id"].astype(str)
test_meta = test_meta.drop_duplicates(subset=["case_id"], keep="first").set_index("case_id")

test_list = man_test.copy()
test_list["case_id"] = test_list["case_id"].astype(str)
overlap_cols = [c for c in test_meta.columns if c in test_list.columns]
if overlap_cols:
    test_list = test_list.drop(columns=overlap_cols)
test_list = test_list.join(test_meta, on="case_id")

sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in train_list.columns]
train_list = train_list.sort_values(sort_train_cols).reset_index(drop=True)
test_list  = test_list.sort_values(["case_id"]).reset_index(drop=True)

if CFG_MATCH["limit_train"] is not None:
    train_list = train_list.iloc[:int(CFG_MATCH["limit_train"])].copy()
if CFG_MATCH["limit_test"] is not None:
    test_list = test_list.iloc[:int(CFG_MATCH["limit_test"])].copy()

print(f"\nTo process:")
print(f"  train samples: {len(train_list):,} (per sample_id)")
print(f"  test  cases  : {len(test_list):,} (per case_id)")

# ----------------------------
# Run matching
# ----------------------------
def run_block(df_list: pd.DataFrame, id_col: str, out_dir: Path, label: str):
    t0 = time.time()
    done = skipped = failed = 0

    for _, row in df_list.iterrows():
        uid = str(row[id_col])
        feat_path = str(row["feat_path"])
        outp = out_dir / f"{uid}.npz"

        if CFG_MATCH["skip_if_exists"] and outp.exists():
            skipped += 1
            continue

        loaded = load_dino_patch_from_npz(feat_path)
        if loaded is None:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN missing/invalid feat uid={uid} feat_path={feat_path}")
            continue

        feat_np, gh, gw = loaded

        try:
            payload = robust_match_one(feat_np, gh, gw, CFG_MATCH)

            payload["uid"] = np.array(uid, dtype=str)
            if "case_id" in row and not pd.isna(row.get("case_id", None)):
                payload["case_id"] = np.array(str(row.get("case_id", "")), dtype=str)
            if "variant" in row and not pd.isna(row.get("variant", None)):
                payload["variant"] = np.array(str(row.get("variant", "")), dtype=str)

            save_match_npz(outp, payload)
            done += 1

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")

        if (done % int(CFG_MATCH["print_every"])) == 0 and done > 0:
            dt = time.time() - t0
            rate = done / max(dt, 1e-9)
            print(f"[{label}] done={done:,} skipped={skipped:,} failed={failed:,} | {rate:.2f} img/s | last={uid}")
            gc.collect()

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done   : {done:,}")
    print(f"  skipped: {skipped:,}")
    print(f"  failed : {failed:,}")
    print(f"  time_s : {dt:.1f}")
    return {"done": done, "skipped": skipped, "failed": failed, "time_s": dt}

print("\n[1/2] Matching TRAIN_ALL ...")
rep_train = run_block(train_list, id_col="sample_id", out_dir=MATCH_TRAIN_DIR, label="train_all")

print("\n[2/2] Matching TEST ...")
rep_test = run_block(test_list, id_col="case_id", out_dir=MATCH_TEST_DIR, label="test")

# ----------------------------
# Write manifests for Stage 4
# ----------------------------
man_match_train = pd.DataFrame({
    "sample_id": train_list["sample_id"].astype(str).values,
    "case_id": train_list["case_id"].astype(str).fillna("").values if "case_id" in train_list.columns else np.array([""]*len(train_list)),
    "variant": train_list["variant"].astype(str).fillna("").values if "variant" in train_list.columns else np.array([""]*len(train_list)),
    "fold": train_list["fold"].fillna(-1).astype(int).values if "fold" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "y_forged": train_list["y_forged"].fillna(-1).astype(int).values if "y_forged" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "match_path": [str(MATCH_TRAIN_DIR / f"{sid}.npz") for sid in train_list["sample_id"].astype(str).values],
})
man_match_train["match_exists"] = man_match_train["match_path"].map(lambda p: Path(p).exists()).astype(int)

man_match_test = pd.DataFrame({
    "case_id": test_list["case_id"].astype(str).values,
    "match_path": [str(MATCH_TEST_DIR / f"{cid}.npz") for cid in test_list["case_id"].astype(str).values],
})
man_match_test["match_exists"] = man_match_test["match_path"].map(lambda p: Path(p).exists()).astype(int)

MATCH_MAN_TRAIN_PATH = MATCH_ROOT / "manifest_match_train_all.csv"
MATCH_MAN_TEST_PATH  = MATCH_ROOT / "manifest_match_test.csv"
man_match_train.to_csv(MATCH_MAN_TRAIN_PATH, index=False)
man_match_test.to_csv(MATCH_MAN_TEST_PATH, index=False)

with open(MATCH_ROOT / "match_summary.json", "w") as f:
    json.dump({"cfg": CFG_MATCH, "train": rep_train, "test": rep_test}, f, indent=2)

print("\nWrote match manifests:")
print(" -", MATCH_MAN_TRAIN_PATH, "| exists_rate:", float(man_match_train["match_exists"].mean()) if len(man_match_train) else 0.0)
print(" -", MATCH_MAN_TEST_PATH,  "| exists_rate:", float(man_match_test["match_exists"].mean()) if len(man_match_test) else 0.0)
print(" -", MATCH_ROOT / "match_summary.json")

MATCH_CACHE_ROOT = str(MATCH_ROOT)
print("\nDONE. Exported: CFG_MATCH, MATCH_CACHE_ROOT, MATCH_TRAIN_DIR, MATCH_TEST_DIR, MATCH_MAN_TRAIN_PATH, MATCH_MAN_TEST_PATH")


MAN_TRAIN_PATH: /kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv | rows(feat_exists=1): 5176
MAN_TEST_PATH : /kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv | rows(feat_exists=1): 1
MATCH_ROOT     : /kaggle/working/recodai_luc/cache/match_base_v3
MATCH_TRAIN_DIR: /kaggle/working/recodai_luc/cache/match_base_v3/train_all
MATCH_TEST_DIR : /kaggle/working/recodai_luc/cache/match_base_v3/test

To process:
  train samples: 5,176 (per sample_id)
  test  cases  : 1 (per case_id)

[1/2] Matching TRAIN_ALL ...
[train_all] done=200 skipped=0 failed=0 | 34.49 img/s | last=3110__auth
[train_all] done=400 skipped=0 failed=0 | 32.31 img/s | last=57390__auth
[train_all] done=600 skipped=0 failed=0 | 32.26 img/s | last=25570__auth
[train_all] done=800 skipped=0 failed=0 | 32.12 img/s | last=50158__auth
[train_all] done=1,000 skipped=0 failed=0 | 32.12 img/s | last=15579__auth
[train_all] done=1,200 skipped=0 failed=0 | 32.58 img/s | last=41583__auth
[train_all] do

# Verification, Mask Reconstruction & Postprocess (One Block)

In [4]:
# ============================================================
# STAGE 4 — Verification, Mask Reconstruction & Postprocess (ONE CELL, REVISI FULL v5 - High Precision)
# English stage title, Indonesian explanations.
#
# Fokus peningkatan:
# - Mask tidak melebar: threshold adaptif + count gating + area guard
# - Skip aman: hanya skip jika cfg_hash sama (kalau cfg berubah -> recompute)
# - Tetap output folder versioned (Opsi A): pred_base_v2_v5
#
# Output:
# - /kaggle/working/recodai_luc/cache/pred_base_v2_v5/train_all/{sample_id}.npz
# - /kaggle/working/recodai_luc/cache/pred_base_v2_v5/test/{case_id}.npz
# - manifest_pred_train_all.csv, manifest_pred_test.csv, pred_summary.json
# - df_pred_feat_train_all
# ============================================================

import os, gc, time, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# morphology / connected components (fallback jika tidak ada scipy)
try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# REQUIRE
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu (df_train_all & df_test).")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

# ----------------------------
# Helper: normalize id -> string stabil (hindari '5506.0')
# ----------------------------
def _norm_one_id(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    if isinstance(x, (np.integer, int)):
        return str(int(x))
    if isinstance(x, (np.floating, float)):
        if np.isfinite(x) and abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(float(x))
    s = str(x)
    if s.endswith(".0"):
        head = s[:-2]
        if head.isdigit():
            return head
    return s

def norm_id_series(s: pd.Series) -> pd.Series:
    return s.map(_norm_one_id)

for c in ["sample_id", "case_id"]:
    if c in df_train_all.columns:
        df_train_all[c] = norm_id_series(df_train_all[c])
if "case_id" in df_test.columns:
    df_test["case_id"] = norm_id_series(df_test["case_id"])

# ----------------------------
# Locate manifests from STAGE 2
# ----------------------------
MAN_TRAIN_PATH = Path(str(globals().get("MAN_TRAIN_PATH", "/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv")))
MAN_TEST_PATH  = Path(str(globals().get("MAN_TEST_PATH",  "/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv")))

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train STAGE 2 tidak ditemukan: {MAN_TRAIN_PATH}")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test STAGE 2 tidak ditemukan: {MAN_TEST_PATH}")

man_train = pd.read_csv(MAN_TRAIN_PATH)
man_test  = pd.read_csv(MAN_TEST_PATH)

need2_train = {"sample_id","feat_path","feat_exists"}
need2_test  = {"case_id","feat_path","feat_exists"}
if not need2_train.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all missing: {need2_train - set(man_train.columns)}")
if not need2_test.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test missing: {need2_test - set(man_test.columns)}")

man_train["sample_id"] = norm_id_series(man_train["sample_id"])
man_test["case_id"] = norm_id_series(man_test["case_id"])
man_train = man_train[man_train["feat_exists"] == 1].copy()
man_test  = man_test[man_test["feat_exists"]  == 1].copy()

# ----------------------------
# Locate manifests from STAGE 3
# - prefer global MATCH_MAN_* (kalau kamu run STAGE 3 v4, ini biasanya sudah v3)
# - fallback: coba v3 lalu v2
# ----------------------------
def _pick_existing(*paths):
    for p in paths:
        p = Path(str(p))
        if p.exists():
            return p
    return Path(str(paths[0]))

MATCH_MAN_TRAIN_PATH = globals().get("MATCH_MAN_TRAIN_PATH", None)
MATCH_MAN_TEST_PATH  = globals().get("MATCH_MAN_TEST_PATH", None)

if MATCH_MAN_TRAIN_PATH is None or MATCH_MAN_TEST_PATH is None:
    MATCH_MAN_TRAIN_PATH = _pick_existing(
        "/kaggle/working/recodai_luc/cache/match_base_v3/manifest_match_train_all.csv",
        "/kaggle/working/recodai_luc/cache/match_base_v2/manifest_match_train_all.csv",
    )
    MATCH_MAN_TEST_PATH = _pick_existing(
        "/kaggle/working/recodai_luc/cache/match_base_v3/manifest_match_test.csv",
        "/kaggle/working/recodai_luc/cache/match_base_v2/manifest_match_test.csv",
    )
else:
    MATCH_MAN_TRAIN_PATH = Path(str(MATCH_MAN_TRAIN_PATH))
    MATCH_MAN_TEST_PATH  = Path(str(MATCH_MAN_TEST_PATH))

if not MATCH_MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest match train STAGE 3 tidak ditemukan: {MATCH_MAN_TRAIN_PATH}")
if not MATCH_MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest match test STAGE 3 tidak ditemukan: {MATCH_MAN_TEST_PATH}")

man_match_train = pd.read_csv(MATCH_MAN_TRAIN_PATH)
man_match_test  = pd.read_csv(MATCH_MAN_TEST_PATH)

need3_train = {"sample_id","match_path","match_exists"}
need3_test  = {"case_id","match_path","match_exists"}
if not need3_train.issubset(set(man_match_train.columns)):
    raise RuntimeError(f"manifest_match_train_all missing: {need3_train - set(man_match_train.columns)}")
if not need3_test.issubset(set(man_match_test.columns)):
    raise RuntimeError(f"manifest_match_test missing: {need3_test - set(man_match_test.columns)}")

man_match_train["sample_id"] = norm_id_series(man_match_train["sample_id"])
man_match_test["case_id"] = norm_id_series(man_match_test["case_id"])

# ----------------------------
# Output dirs (OPS I A: versioned)
# ----------------------------
PRED_ROOT = Path("/kaggle/working/recodai_luc/cache/pred_base_v2_v5")
PRED_TRAIN_DIR = PRED_ROOT / "train_all"
PRED_TEST_DIR  = PRED_ROOT / "test"
PRED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PRED_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows(feat_exists=1):", len(man_train))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows(feat_exists=1):", len(man_test))
print("MATCH_MAN_TRAIN_PATH:", MATCH_MAN_TRAIN_PATH, "| rows:", len(man_match_train))
print("MATCH_MAN_TEST_PATH :", MATCH_MAN_TEST_PATH,  "| rows:", len(man_match_test))
print("PRED_ROOT:", PRED_ROOT)
print("SCIPY available:", _HAS_SCIPY)

# ----------------------------
# CONFIG (lebih ketat agar mask tidak melebar)
# ----------------------------
CFG_RECON = {
    # verification (disimpan sebagai fitur gate juga)
    "sim_inlier_thr": 0.82,
    "min_pairs": 25,

    # build grid from pairs (score+count)
    "score_mix": 0.65,          # lebih berat ke score tapi count tetap dipakai
    "grid_thr_q": 0.90,         # threshold quantile untuk comb score
    "min_count_q": 0.80,        # minimal count quantile (patch harus sering muncul di pairs)
    "thr_grid_floor": 0.40,     # floor supaya tidak terlalu longgar
    "grid_dilate": 0,           # 0 = tidak melebarkan di grid (lebih precision)

    # postprocess (orig resolution)
    "do_open": True,
    "open_ks": 3,
    "do_close": True,
    "close_ks": 3,

    # component filter
    "min_area_frac": 0.0006,    # buang noise kecil
    "keep_topk_components": 2,  # jangan banyak komponen

    # hard guard: kalau mask terlalu besar, biasanya FP -> kosongkan
    "max_area_frac": 0.35,      # aman, bisa turunkan ke 0.25 kalau masih sering overmask

    # io
    "skip_if_exists": True,     # skip hanya jika cfg_hash sama
    "print_every": 400,
}

_CFG_HASH = hashlib.md5(json.dumps(CFG_RECON, sort_keys=True).encode()).hexdigest()[:10]

# ----------------------------
# Helpers
# ----------------------------
def _meta_to_str(x):
    if isinstance(x, np.ndarray):
        try:
            x = x.item()
        except Exception:
            x = x.reshape(-1)[0]
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8", errors="ignore")
    return str(x)

def _read_meta_from_feat_npz(feat_path: str):
    p = Path(feat_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    if "meta" not in z.files:
        return None
    try:
        meta = json.loads(_meta_to_str(z["meta"]))
    except Exception:
        return None

    oh, ow = meta.get("orig_hw", [None, None])
    rh, rw = meta.get("resized_hw", [None, None])
    gh, gw = meta.get("grid_hw", [None, None])
    patch = meta.get("patch_size", None)
    if None in [oh, ow, rh, rw, gh, gw, patch]:
        return None
    return {
        "orig_h": int(oh), "orig_w": int(ow),
        "proc_h": int(rh), "proc_w": int(rw),
        "grid_h": int(gh), "grid_w": int(gw),
        "patch": int(patch),
    }

def _load_match_npz(match_path: str):
    p = Path(match_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)

    out = {}
    for k in ["grid_h","grid_w","has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
              "n_pairs_thr","n_pairs_mnn","best_src","best_dst","best_sim"]:
        if k in z.files:
            out[k] = z[k]

    def _s(v, default=0):
        if v is None:
            return default
        if isinstance(v, np.ndarray):
            if np.ndim(v) == 0:
                return v.item()
            return v.reshape(-1)[0].item()
        return v

    out["grid_h"] = int(_s(out.get("grid_h", None), 0))
    out["grid_w"] = int(_s(out.get("grid_w", None), 0))
    out["has_peak"] = int(_s(out.get("has_peak", None), 0))
    out["peak_ratio"] = float(_s(out.get("peak_ratio", None), 0.0))
    out["best_weight"] = float(_s(out.get("best_weight", None), 0.0))
    out["best_count"] = int(_s(out.get("best_count", None), 0))
    out["best_mean_sim"] = float(_s(out.get("best_mean_sim", None), 0.0))
    out["n_pairs_thr"] = int(_s(out.get("n_pairs_thr", None), 0))
    out["n_pairs_mnn"] = int(_s(out.get("n_pairs_mnn", None), 0))

    out["best_src"] = out.get("best_src", np.zeros((0,), dtype=np.int32)).astype(np.int32, copy=False)
    out["best_dst"] = out.get("best_dst", np.zeros((0,), dtype=np.int32)).astype(np.int32, copy=False)
    out["best_sim"] = out.get("best_sim", np.zeros((0,), dtype=np.float16)).astype(np.float16, copy=False)
    return out

def pack_mask(mask_u8: np.ndarray) -> np.ndarray:
    m = (mask_u8 > 0).astype(np.uint8, copy=False).reshape(-1)
    return np.packbits(m, axis=None)

def _binary_close(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_closing(mask.astype(bool), structure=st).astype(np.uint8)

def _binary_open(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_opening(mask.astype(bool), structure=st).astype(np.uint8)

def _filter_components(mask: np.ndarray, min_area: int, keep_topk: int):
    mask = mask.astype(np.uint8, copy=False)
    if mask.sum() == 0:
        return mask, 0, 0

    if not _HAS_SCIPY:
        area = int(mask.sum())
        if area < int(min_area):
            return np.zeros_like(mask, dtype=np.uint8), 0, 0
        return mask, 1, area

    lab, n = ndi.label(mask.astype(bool))
    if n == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    areas = np.bincount(lab.ravel())
    areas[0] = 0

    comps = np.where(areas >= int(min_area))[0]
    if comps.size == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    comps = comps[np.argsort(areas[comps])[::-1]]
    if keep_topk is not None and int(keep_topk) > 0:
        comps = comps[:int(keep_topk)]

    out = np.isin(lab, comps).astype(np.uint8)
    largest = int(areas[comps[0]]) if comps.size else 0
    return out, int(comps.size), largest

def build_grid_from_pairs_hp(best_src, best_dst, best_sim, gh, gw, cfg):
    """
    High-precision grid building:
    - patch_score: akumulasi sim ke endpoint (src+dst)
    - patch_count: frekuensi endpoint muncul
    - comb = mix*score_norm + (1-mix)*count_norm
    - threshold adaptif: max(thr_floor, quantile(comb_nonzero, q))
    - count gate adaptif: count >= quantile(count_nonzero, min_count_q)
    """
    N = int(gh * gw)
    if best_src is None or best_src.size == 0 or N <= 0:
        grid_score = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
        grid_count = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint16)
        mask_grid  = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint8)
        feats = {"inlier_ratio": 0.0, "pair_count": 0, "uniq_src": 0, "uniq_dst": 0, "mean_sim": 0.0,
                 "thr_used": 1.0, "cnt_thr_used": 999999}
        return grid_score, grid_count, mask_grid, feats

    src = best_src.astype(np.int32, copy=False)
    dst = best_dst.astype(np.int32, copy=False)
    sim = best_sim.astype(np.float32, copy=False)

    patch_score = np.zeros((N,), dtype=np.float32)
    patch_count = np.zeros((N,), dtype=np.int32)

    w = np.clip(sim, 0.0, 1.0)
    np.add.at(patch_score, src, w); np.add.at(patch_score, dst, w)
    np.add.at(patch_count, src, 1); np.add.at(patch_count, dst, 1)

    smax = float(patch_score.max()) if patch_score.size else 0.0
    score_norm = patch_score / (smax + 1e-9) if smax > 0 else patch_score

    cmax = float(patch_count.max()) if patch_count.size else 0.0
    count_norm = patch_count.astype(np.float32) / (cmax + 1e-9) if cmax > 0 else patch_count.astype(np.float32)

    mix = float(cfg["score_mix"])
    comb = mix * score_norm + (1.0 - mix) * count_norm

    comb_nz = comb[comb > 0]
    if comb_nz.size > 0:
        thr_dyn = float(np.quantile(comb_nz, float(cfg["grid_thr_q"])))
    else:
        thr_dyn = 1.0
    thr_used = max(float(cfg["thr_grid_floor"]), thr_dyn)

    cnt_nz = patch_count[patch_count > 0]
    if cnt_nz.size > 0:
        cnt_thr = int(np.quantile(cnt_nz.astype(np.float32), float(cfg["min_count_q"])))
        cnt_thr = max(1, cnt_thr)
    else:
        cnt_thr = 999999

    mask_flat = (comb >= thr_used) & (patch_count >= cnt_thr)
    mask_grid = mask_flat.reshape(gh, gw).astype(np.uint8)

    grid_score = comb.reshape(gh, gw).astype(np.float32)
    grid_count = patch_count.reshape(gh, gw).astype(np.uint16)

    inlier_thr = float(cfg["sim_inlier_thr"])
    inlier_ratio = float((sim >= inlier_thr).mean()) if sim.size else 0.0

    feats = {
        "inlier_ratio": inlier_ratio,
        "pair_count": int(sim.size),
        "uniq_src": int(np.unique(src).size),
        "uniq_dst": int(np.unique(dst).size),
        "mean_sim": float(sim.mean()) if sim.size else 0.0,
        "thr_used": float(thr_used),
        "cnt_thr_used": int(cnt_thr),
    }
    return grid_score, grid_count, mask_grid, feats

def grid_to_orig(mask_grid, patch, proc_h, proc_w, orig_h, orig_w):
    patch = max(int(patch), 1)
    mask_proc = np.kron(mask_grid.astype(np.uint8), np.ones((patch, patch), dtype=np.uint8))
    mask_proc = mask_proc[:max(proc_h,1), :max(proc_w,1)]
    if (proc_h, proc_w) != (orig_h, orig_w):
        im = Image.fromarray((mask_proc * 255).astype(np.uint8))
        im = im.resize((max(orig_w,1), max(orig_h,1)), resample=Image.NEAREST)
        mask_orig = (np.array(im) > 0).astype(np.uint8)
    else:
        mask_orig = mask_proc.astype(np.uint8)
    return mask_orig

def _as_save_scalar(v):
    if isinstance(v, (bool, np.bool_)):
        return np.int8(int(v))
    if isinstance(v, (int, np.integer)):
        return np.int32(int(v))
    if isinstance(v, (float, np.floating)):
        return np.float32(float(v))
    try:
        return np.float32(float(v))
    except Exception:
        return np.float32(0.0)

def save_pred_npz(out_path: Path, mask_orig: np.ndarray, grid_score: np.ndarray, grid_count: np.ndarray, scalars: dict):
    mask_pack = pack_mask(mask_orig)
    payload = dict(
        mask_pack=mask_pack.astype(np.uint8),
        mask_h=np.int32(mask_orig.shape[0]),
        mask_w=np.int32(mask_orig.shape[1]),
        grid_score=grid_score.astype(np.float16, copy=False),
        grid_count=grid_count.astype(np.uint16, copy=False),
        cfg_hash=np.array(_CFG_HASH),
    )
    for k, v in scalars.items():
        payload[k] = _as_save_scalar(v)
    np.savez_compressed(str(out_path), **payload)

def load_pred_scalars_from_npz(pred_path: Path):
    if not pred_path.exists():
        return None
    z = np.load(pred_path, allow_pickle=False)
    out = {}
    for k in ["has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
              "inlier_ratio","pair_count","uniq_src","uniq_dst","mean_sim",
              "area_frac","n_comp","largest_comp","n_pairs_thr","n_pairs_mnn",
              "thr_used","cnt_thr_used","mask_thr_used","sim_inlier_thr_used"]:
        if k in z.files:
            v = z[k]
            if np.ndim(v) == 0:
                out[k] = float(v)
            else:
                out[k] = float(np.array(v).reshape(-1)[0])
        else:
            out[k] = 0.0
    return out

def pred_cfg_hash_matches(pred_path: Path) -> bool:
    if not pred_path.exists():
        return False
    try:
        z = np.load(pred_path, allow_pickle=False)
        if "cfg_hash" not in z.files:
            return False
        h = z["cfg_hash"]
        if isinstance(h, np.ndarray):
            try:
                h = h.item()
            except Exception:
                h = str(h.reshape(-1)[0])
        h = str(h)
        return h == str(_CFG_HASH)
    except Exception:
        return False

# ----------------------------
# Build worklists
# ----------------------------
work_train = man_match_train.merge(man_train[["sample_id","feat_path"]], on="sample_id", how="left")

train_need_cols = ["case_id","variant","fold","y_forged","image_path"]
missing_cols = [c for c in train_need_cols if c not in work_train.columns]
if missing_cols:
    take = ["sample_id"] + [c for c in missing_cols if c in df_train_all.columns]
    work_train = work_train.merge(df_train_all[take], on="sample_id", how="left")

work_train["sample_id"] = norm_id_series(work_train["sample_id"])
if "case_id" in work_train.columns:
    work_train["case_id"] = norm_id_series(work_train["case_id"])
work_train["feat_path"]  = work_train["feat_path"].fillna("").astype(str)
work_train["match_path"] = work_train["match_path"].fillna("").astype(str)

work_test = man_match_test.merge(man_test[["case_id","feat_path"]], on="case_id", how="left")
if "image_path" not in work_test.columns:
    if "case_id" in df_test.columns and "image_path" in df_test.columns:
        tmp = df_test[["case_id","image_path"]].copy()
        tmp["case_id"] = norm_id_series(tmp["case_id"])
        work_test = work_test.merge(tmp, on="case_id", how="left")

work_test["case_id"] = norm_id_series(work_test["case_id"])
work_test["feat_path"]  = work_test["feat_path"].fillna("").astype(str)
work_test["match_path"] = work_test["match_path"].fillna("").astype(str)

sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in work_train.columns]
work_train = work_train.sort_values(sort_train_cols).reset_index(drop=True) if sort_train_cols else work_train.sort_values(["sample_id"]).reset_index(drop=True)
work_test  = work_test.sort_values(["case_id"]).reset_index(drop=True)

print("\nWork sizes:")
print("  train_all:", len(work_train), "| match_exists rate:", float(work_train["match_exists"].mean()) if len(work_train) else 0.0)
print("  test     :", len(work_test),  "| match_exists rate:", float(work_test["match_exists"].mean()) if len(work_test) else 0.0)

# ----------------------------
# Main loop
# ----------------------------
def run_pred_block(dfw: pd.DataFrame, id_col: str, out_dir: Path, label: str, collect_feat_rows: bool):
    t0 = time.time()
    done = skipped = failed = 0
    feat_rows = []

    for _, row in dfw.iterrows():
        uid = str(row[id_col])
        outp = out_dir / f"{uid}.npz"

        # Skip only if file exists AND cfg_hash matches current cfg
        if CFG_RECON["skip_if_exists"] and outp.exists() and pred_cfg_hash_matches(outp):
            skipped += 1
            if collect_feat_rows:
                scal = load_pred_scalars_from_npz(outp) or {}
                feat_row = {"uid": uid, **scal}
                for c in ["case_id","variant","fold","y_forged"]:
                    if c in row and not pd.isna(row.get(c, None)):
                        feat_row[c] = row.get(c)
                feat_rows.append(feat_row)
            continue

        feat_path  = str(row.get("feat_path",""))
        match_path = str(row.get("match_path",""))

        meta = _read_meta_from_feat_npz(feat_path) if feat_path else None
        mch  = _load_match_npz(match_path) if match_path else None

        if meta is None:
            ip = str(row.get("image_path",""))
            if ip and Path(ip).exists():
                im = Image.open(ip).convert("RGB")
                ow, oh = im.size
                meta = {"orig_h": int(oh), "orig_w": int(ow),
                        "proc_h": int(oh), "proc_w": int(ow),
                        "grid_h": 1, "grid_w": 1, "patch": 1}
            else:
                meta = {"orig_h": 1, "orig_w": 1, "proc_h": 1, "proc_w": 1, "grid_h": 1, "grid_w": 1, "patch": 1}

        orig_h, orig_w = int(meta["orig_h"]), int(meta["orig_w"])
        gh, gw         = int(meta["grid_h"]), int(meta["grid_w"])
        proc_h, proc_w = int(meta["proc_h"]), int(meta["proc_w"])
        patch          = int(meta["patch"])

        mask_orig  = np.zeros((max(orig_h,1), max(orig_w,1)), dtype=np.uint8)
        grid_score = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
        grid_count = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint16)

        scalars = {
            "has_peak": 0,
            "peak_ratio": 0.0,
            "best_weight": 0.0,
            "best_count": 0,
            "best_mean_sim": 0.0,
            "inlier_ratio": 0.0,
            "pair_count": 0,
            "uniq_src": 0,
            "uniq_dst": 0,
            "mean_sim": 0.0,
            "area_frac": 0.0,
            "n_comp": 0,
            "largest_comp": 0,
            "n_pairs_thr": 0,
            "n_pairs_mnn": 0,
            "thr_used": 0.0,
            "cnt_thr_used": 0.0,
            "mask_thr_used": float(CFG_RECON["thr_grid_floor"]),
            "sim_inlier_thr_used": float(CFG_RECON["sim_inlier_thr"]),
        }

        try:
            if mch is not None:
                gh_m = int(mch.get("grid_h", gh))
                gw_m = int(mch.get("grid_w", gw))
                if (gh_m > 0 and gw_m > 0) and (gh_m != gh or gw_m != gw):
                    gh, gw = gh_m, gw_m
                    grid_score = np.zeros((gh, gw), dtype=np.float32)
                    grid_count = np.zeros((gh, gw), dtype=np.uint16)

                has_peak   = int(mch.get("has_peak", 0))
                best_count = int(mch.get("best_count", 0))

                scalars["has_peak"] = has_peak
                scalars["peak_ratio"] = float(mch.get("peak_ratio", 0.0))
                scalars["best_weight"] = float(mch.get("best_weight", 0.0))
                scalars["best_count"] = int(best_count)
                scalars["best_mean_sim"] = float(mch.get("best_mean_sim", 0.0))
                scalars["n_pairs_thr"] = int(mch.get("n_pairs_thr", 0))
                scalars["n_pairs_mnn"] = int(mch.get("n_pairs_mnn", 0))

                if has_peak == 1 and best_count >= int(CFG_RECON["min_pairs"]):
                    best_src = mch.get("best_src", np.zeros((0,), dtype=np.int32))
                    best_dst = mch.get("best_dst", np.zeros((0,), dtype=np.int32))
                    best_sim = mch.get("best_sim", np.zeros((0,), dtype=np.float16))

                    grid_score, grid_count, mask_grid, vfeats = build_grid_from_pairs_hp(
                        best_src, best_dst, best_sim, gh, gw, CFG_RECON
                    )

                    mask_orig = grid_to_orig(mask_grid, patch, proc_h, proc_w, orig_h, orig_w)

                    # postprocess
                    if CFG_RECON["do_open"]:
                        mask_orig = _binary_open(mask_orig, int(CFG_RECON["open_ks"]))
                    if CFG_RECON["do_close"]:
                        mask_orig = _binary_close(mask_orig, int(CFG_RECON["close_ks"]))

                    min_area = int(float(CFG_RECON["min_area_frac"]) * float(mask_orig.size))
                    if min_area < 1:
                        min_area = 1
                    mask_orig, n_comp, largest = _filter_components(
                        mask_orig, min_area=min_area, keep_topk=int(CFG_RECON["keep_topk_components"])
                    )

                    area_frac = float(mask_orig.sum()) / float(mask_orig.size + 1e-9)

                    # hard guard anti-overmask
                    if area_frac > float(CFG_RECON["max_area_frac"]):
                        mask_orig[:] = 0
                        area_frac = 0.0
                        n_comp = 0
                        largest = 0

                    scalars["inlier_ratio"] = float(vfeats["inlier_ratio"])
                    scalars["pair_count"]   = int(vfeats["pair_count"])
                    scalars["uniq_src"]     = int(vfeats["uniq_src"])
                    scalars["uniq_dst"]     = int(vfeats["uniq_dst"])
                    scalars["mean_sim"]     = float(vfeats["mean_sim"])
                    scalars["area_frac"]    = float(area_frac)
                    scalars["n_comp"]       = int(n_comp)
                    scalars["largest_comp"] = int(largest)
                    scalars["thr_used"]     = float(vfeats.get("thr_used", 0.0))
                    scalars["cnt_thr_used"] = float(vfeats.get("cnt_thr_used", 0))

            save_pred_npz(outp, mask_orig, grid_score, grid_count, scalars)

            if collect_feat_rows:
                feat_row = {"uid": uid, **scalars}
                for c in ["case_id","variant","fold","y_forged"]:
                    if c in row and not pd.isna(row.get(c, None)):
                        feat_row[c] = row.get(c)
                feat_rows.append(feat_row)

            done += 1
            if (done % int(CFG_RECON["print_every"])) == 0 and done > 0:
                dt = time.time() - t0
                print(f"[{label}] done={done:,} skipped={skipped:,} failed={failed:,} | {done/max(dt,1e-9):.2f} item/s | last={uid}")
                gc.collect()

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done   : {done:,}")
    print(f"  skipped: {skipped:,}")
    print(f"  failed : {failed:,}")
    print(f"  time_s : {dt:.1f}")
    return feat_rows, {"done": done, "skipped": skipped, "failed": failed, "time_s": dt}

print("\n[1/2] Build preds for TRAIN_ALL (per sample_id) ...")
feat_rows_train, rep_train = run_pred_block(work_train, "sample_id", PRED_TRAIN_DIR, "train_all", collect_feat_rows=True)

print("\n[2/2] Build preds for TEST (per case_id) ...")
feat_rows_test, rep_test = run_pred_block(work_test, "case_id", PRED_TEST_DIR, "test", collect_feat_rows=False)

# ----------------------------
# Write pred manifests
# ----------------------------
man_pred_train = pd.DataFrame({
    "sample_id": work_train["sample_id"].astype(str).values,
    "case_id": work_train["case_id"].astype(str).fillna("").values if "case_id" in work_train.columns else [""]*len(work_train),
    "variant": work_train["variant"].astype(str).fillna("").values if "variant" in work_train.columns else [""]*len(work_train),
    "fold": work_train["fold"].fillna(-1).astype(int).values if "fold" in work_train.columns else [-1]*len(work_train),
    "y_forged": work_train["y_forged"].fillna(-1).astype(int).values if "y_forged" in work_train.columns else [-1]*len(work_train),
    "pred_path": [str(PRED_TRAIN_DIR / f"{sid}.npz") for sid in work_train["sample_id"].astype(str).values],
})
man_pred_train["pred_exists"] = man_pred_train["pred_path"].map(lambda p: Path(p).exists()).astype(int)

man_pred_test = pd.DataFrame({
    "case_id": work_test["case_id"].astype(str).values,
    "pred_path": [str(PRED_TEST_DIR / f"{cid}.npz") for cid in work_test["case_id"].astype(str).values],
})
man_pred_test["pred_exists"] = man_pred_test["pred_path"].map(lambda p: Path(p).exists()).astype(int)

PRED_MAN_TRAIN_PATH = PRED_ROOT / "manifest_pred_train_all.csv"
PRED_MAN_TEST_PATH  = PRED_ROOT / "manifest_pred_test.csv"
man_pred_train.to_csv(PRED_MAN_TRAIN_PATH, index=False)
man_pred_test.to_csv(PRED_MAN_TEST_PATH, index=False)

with open(PRED_ROOT / "pred_summary.json", "w") as f:
    json.dump({"cfg": CFG_RECON, "cfg_hash": _CFG_HASH, "train": rep_train, "test": rep_test}, f, indent=2)

df_pred_feat_train_all = pd.DataFrame(feat_rows_train) if len(feat_rows_train) else pd.DataFrame()

print("\nWrote pred manifests:")
print(" -", PRED_MAN_TRAIN_PATH, "| exists_rate:", float(man_pred_train["pred_exists"].mean()) if len(man_pred_train) else 0.0)
print(" -", PRED_MAN_TEST_PATH,  "| exists_rate:", float(man_pred_test["pred_exists"].mean()) if len(man_pred_test) else 0.0)
print(" -", PRED_ROOT / "pred_summary.json")

PRED_CACHE_ROOT = str(PRED_ROOT)
PRED_TRAIN_CACHE_DIR = str(PRED_TRAIN_DIR)
PRED_TEST_CACHE_DIR  = str(PRED_TEST_DIR)

print("\nDONE. Exported: CFG_RECON, PRED_CACHE_ROOT, PRED_TRAIN_CACHE_DIR, PRED_TEST_CACHE_DIR, PRED_MAN_TRAIN_PATH, PRED_MAN_TEST_PATH")
print("df_pred_feat_train_all head:")
print(df_pred_feat_train_all.head())


MAN_TRAIN_PATH: /kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv | rows(feat_exists=1): 5176
MAN_TEST_PATH : /kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv | rows(feat_exists=1): 1
MATCH_MAN_TRAIN_PATH: /kaggle/working/recodai_luc/cache/match_base_v3/manifest_match_train_all.csv | rows: 5176
MATCH_MAN_TEST_PATH : /kaggle/working/recodai_luc/cache/match_base_v3/manifest_match_test.csv | rows: 1
PRED_ROOT: /kaggle/working/recodai_luc/cache/pred_base_v2_v5
SCIPY available: True

Work sizes:
  train_all: 5176 | match_exists rate: 1.0
  test     : 1 | match_exists rate: 1.0

[1/2] Build preds for TRAIN_ALL (per sample_id) ...
[train_all] done=400 skipped=0 failed=0 | 24.41 item/s | last=57390__auth
[train_all] done=800 skipped=0 failed=0 | 22.41 item/s | last=50158__auth
[train_all] done=1,200 skipped=0 failed=0 | 22.41 item/s | last=41583__auth
[train_all] done=1,600 skipped=0 failed=0 | 21.99 item/s | last=31329__auth
[train_all] done=2,000 skipped=0

# Train & Save: Gate Model + Calibration + Thresholds

In [5]:
# ============================================================
# STAGE 5 — Train & Save: Gate Model + Calibration + Thresholds (ONE CELL, REVISI FULL v3 - Stronger Gate)
# English stage title, Indonesian explanations.
#
# Upgrade utama:
# - Ensemble gate: LogisticRegression + ExtraTrees + HistGradientBoosting (non-linear)
# - Tambah fitur pred_area_frac dihitung ulang dari mask_pack (lebih stabil)
# - Threshold tuning pakai mean per-fold (lebih robust)
# - Hilangkan DeprecationWarning Pillow (tidak pakai mode=)
# - Auto-pick pred manifest terbaru (v5/v4/v2) jika PRED_MAN_TRAIN_PATH tidak diset
#
# thresholds.json kompatibel STAGE 6:
# - key: thr_forged (alias thr_p), thr_area, thr_inlier, require_has_peak
# ============================================================

import os, json, time, math, ast
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

import joblib
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

# ----------------------------
# REQUIRE
# ----------------------------
if "df_train_all" not in globals():
    raise RuntimeError("Missing df_train_all. Jalankan STAGE 1 dulu.")

df_train_all = df_train_all.copy()

need_cols = {"sample_id","case_id","variant","fold","y_forged","mask_paths"}
miss = need_cols - set(df_train_all.columns)
if miss:
    raise RuntimeError(f"df_train_all missing columns: {miss}")

df_train_all["sample_id"] = df_train_all["sample_id"].astype(str)
df_train_all["case_id"]   = df_train_all["case_id"].astype(str)
df_train_all["variant"]   = df_train_all["variant"].astype(str)
df_train_all["fold"]      = df_train_all["fold"].astype(int)
df_train_all["y_forged"]  = df_train_all["y_forged"].astype(int)

# ----------------------------
# Pick pred manifest (STAGE 4 output)
# ----------------------------
def _pick_existing(*paths):
    for p in paths:
        p = Path(str(p))
        if p.exists():
            return p
    return None

if "PRED_MAN_TRAIN_PATH" in globals():
    PRED_MAN_TRAIN_PATH = Path(str(PRED_MAN_TRAIN_PATH))
else:
    PRED_MAN_TRAIN_PATH = _pick_existing(
        "/kaggle/working/recodai_luc/cache/pred_base_v2_v5/manifest_pred_train_all.csv",
        "/kaggle/working/recodai_luc/cache/pred_base_v2_v4/manifest_pred_train_all.csv",
        "/kaggle/working/recodai_luc/cache/pred_base_v2/manifest_pred_train_all.csv",
        "/kaggle/working/recodai_luc/cache/pred_base_v2_v2/manifest_pred_train_all.csv",
    )

if PRED_MAN_TRAIN_PATH is None or (not PRED_MAN_TRAIN_PATH.exists()):
    raise FileNotFoundError("Pred manifest tidak ditemukan. Jalankan STAGE 4 dulu (pred manifest).")

man_pred = pd.read_csv(PRED_MAN_TRAIN_PATH)
if not {"sample_id","pred_path","pred_exists"}.issubset(set(man_pred.columns)):
    raise RuntimeError("manifest_pred_train_all.csv harus punya: sample_id, pred_path, pred_exists")

man_pred["sample_id"] = man_pred["sample_id"].astype(str)
man_pred["pred_path"] = man_pred["pred_path"].astype(str)
man_pred = man_pred[man_pred["pred_exists"] == 1].copy()

# join (hanya yang ada pred file)
df = df_train_all.merge(man_pred[["sample_id","pred_path"]], on="sample_id", how="inner").reset_index(drop=True)

print("PRED_MAN_TRAIN_PATH:", PRED_MAN_TRAIN_PATH)
print(f"Train_all rows (with pred): {len(df):,} / total df_train_all: {len(df_train_all):,}")
print(f"Class rate forged: {df['y_forged'].mean():.4f} | folds: {df['fold'].nunique()}")

# ----------------------------
# POPCOUNT LUT (untuk hitung area dari mask_pack)
# ----------------------------
POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

# ----------------------------
# Load scalar features from pred NPZ (per sample_id)
# - plus: pred_area_frac dihitung ulang dari mask_pack
# ----------------------------
BASE_SCALARS = [
    "has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
    "inlier_ratio","pair_count","uniq_src","uniq_dst",
    "area_frac","n_comp","largest_comp",
    "n_pairs_thr","n_pairs_mnn",
    # optional (kalau ada dari STAGE 4 versi lebih baru)
    "thr_used","cnt_thr_used","mask_thr_used","sim_inlier_thr_used"
]

def _scalar(z, k, default=0.0):
    if k not in z.files:
        return float(default)
    v = z[k]
    if np.ndim(v) == 0:
        return float(v)
    return float(np.array(v).reshape(-1)[0])

def load_pred_row(sample_id: str, pred_path: str):
    p = Path(pred_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)

    mh = int(_scalar(z, "mask_h", 0))
    mw = int(_scalar(z, "mask_w", 0))
    denom = float(mh * mw) if (mh > 0 and mw > 0) else 0.0

    # compute pred_area_frac from mask_pack (lebih kuat untuk gate)
    pred_pack = z["mask_pack"].astype(np.uint8).reshape(-1) if "mask_pack" in z.files else np.zeros((0,), dtype=np.uint8)
    pred_area = int(POPCNT[pred_pack].sum()) if pred_pack.size else 0
    pred_area_frac = (pred_area / denom) if denom > 0 else 0.0

    row = {"sample_id": sample_id, "pred_path": pred_path, "mask_h": mh, "mask_w": mw,
           "pred_area": float(pred_area), "pred_area_frac": float(pred_area_frac)}
    for k in BASE_SCALARS:
        row[k] = _scalar(z, k, 0.0)

    row["largest_frac"] = (float(row["largest_comp"]) / denom) if denom > 0 else 0.0
    return row

feat_rows = []
miss_pred = 0
for sid, pth in zip(df["sample_id"].tolist(), df["pred_path"].tolist()):
    r = load_pred_row(sid, pth)
    if r is None:
        miss_pred += 1
        r = {"sample_id": sid, "pred_path": pth, "mask_h": 0, "mask_w": 0, "pred_area": 0.0, "pred_area_frac": 0.0}
        for k in BASE_SCALARS:
            r[k] = 0.0
        r["largest_frac"] = 0.0
    feat_rows.append(r)

df_feat = pd.DataFrame(feat_rows)
df = df.merge(df_feat, on=["sample_id","pred_path"], how="left")
print(f"Loaded pred scalars. Missing pred files during read: {miss_pred:,}")

# ----------------------------
# Feature engineering
# ----------------------------
def safe_log1p(x):
    x = np.asarray(x, dtype=np.float32)
    return np.log1p(np.clip(x, 0.0, None))

df["peak_ratio_clip"]  = np.clip(df["peak_ratio"].astype(np.float32), 0.0, 80.0)
df["log_best_count"]   = safe_log1p(df["best_count"])
df["log_pair_count"]   = safe_log1p(df["pair_count"])
df["log_n_comp"]       = safe_log1p(df["n_comp"])
df["log_largest"]      = safe_log1p(df["largest_comp"])
df["log_pred_area"]    = safe_log1p(df["pred_area"])
df["sqrt_area"]        = np.sqrt(np.clip(df["pred_area_frac"].astype(np.float32), 0.0, 1.0))

has_thr = ("n_pairs_thr" in df.columns)
has_mnn = ("n_pairs_mnn" in df.columns)
df["log_thr_pairs"] = safe_log1p(df["n_pairs_thr"]) if has_thr else 0.0
df["log_mnn_pairs"] = safe_log1p(df["n_pairs_mnn"]) if has_mnn else 0.0

# One-hot variant (kadang sangat membantu kalau domain shift antar variant)
variant_dum = pd.get_dummies(df["variant"].fillna(""), prefix="v", dummy_na=False)
df = pd.concat([df, variant_dum], axis=1)

NUM_FEATURES = [
    "has_peak",
    "peak_ratio_clip",
    "best_weight",
    "best_mean_sim",
    "inlier_ratio",
    "area_frac",         # scalar dari stage4 (tetap dipakai)
    "pred_area_frac",    # recompute dari mask_pack (lebih kuat)
    "largest_frac",
    "sqrt_area",
    "log_best_count",
    "log_pair_count",
    "log_n_comp",
    "log_largest",
    "log_pred_area",
    "uniq_src",
    "uniq_dst",
]
if has_thr and has_mnn:
    NUM_FEATURES += ["log_thr_pairs", "log_mnn_pairs"]

# optional jika ada
for opt in ["thr_used","cnt_thr_used","mask_thr_used","sim_inlier_thr_used"]:
    if opt in df.columns:
        NUM_FEATURES.append(opt)

FEATURE_COLS = NUM_FEATURES + list(variant_dum.columns)

df[FEATURE_COLS] = df[FEATURE_COLS].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype(np.float32)

X = df[FEATURE_COLS].values.astype(np.float32)
y = df["y_forged"].values.astype(int)
folds = df["fold"].values.astype(int)
uniq_y = np.unique(y)

# balanced weights (dipakai untuk tree dan boosting)
pos = float((y == 1).sum())
neg = float((y == 0).sum())
if pos > 0 and neg > 0:
    w_pos = (pos + neg) / (2.0 * pos)
    w_neg = (pos + neg) / (2.0 * neg)
else:
    w_pos = 1.0; w_neg = 1.0
sample_weight = np.where(y == 1, w_pos, w_neg).astype(np.float32)

# ----------------------------
# Models
# ----------------------------
def make_lr():
    return Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("clf", LogisticRegression(
            solver="liblinear",
            class_weight="balanced",
            max_iter=5000,
            C=1.5,
            random_state=2025
        ))
    ])

def make_et():
    return ExtraTreesClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=3,
        min_samples_split=8,
        max_features="sqrt",
        bootstrap=False,
        class_weight="balanced",
        n_jobs=-1,
        random_state=2025
    )

def make_hgb():
    # HGB biasanya kuat untuk interaksi fitur (non-linear)
    return HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_depth=5,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=0.0,
        max_bins=255,
        random_state=2025
    )

# ----------------------------
# OOF training (ensemble)
# ----------------------------
oof_prob = np.zeros(len(df), dtype=np.float32)
models_by_fold = {}

t0 = time.time()
for f in sorted(np.unique(folds)):
    tr = (folds != f)
    va = (folds == f)
    if va.sum() == 0:
        continue

    lr = make_lr()
    et = make_et()
    hgb = make_hgb()

    lr.fit(X[tr], y[tr])
    et.fit(X[tr], y[tr])
    hgb.fit(X[tr], y[tr], sample_weight=sample_weight[tr])

    p_lr  = lr.predict_proba(X[va])[:, 1].astype(np.float32)
    p_et  = et.predict_proba(X[va])[:, 1].astype(np.float32)
    p_hgb = hgb.predict_proba(X[va])[:, 1].astype(np.float32)

    # weighted average (bias ke model non-linear sedikit)
    p = (0.25 * p_lr + 0.40 * p_et + 0.35 * p_hgb).astype(np.float32)
    oof_prob[va] = p

    models_by_fold[int(f)] = {"lr": lr, "et": et, "hgb": hgb}

dt = time.time() - t0
if len(uniq_y) > 1:
    auc = roc_auc_score(y, oof_prob)
    ap  = average_precision_score(y, oof_prob)
else:
    auc = float("nan"); ap = float("nan")
print(f"OOF ensemble: AUC={auc:.5f} | AP={ap:.5f} | time_s={dt:.1f}")

# ----------------------------
# Calibration on OOF
# ----------------------------
calibrator = None
calib_kind = "none"

try:
    if len(uniq_y) >= 2 and len(y) >= 200:
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(oof_prob, y)
        calibrator = iso
        calib_kind = "isotonic"
    else:
        raise RuntimeError("Not enough data for isotonic.")
except Exception:
    try:
        platt = LogisticRegression(solver="lbfgs", max_iter=2000)
        platt.fit(oof_prob.reshape(-1,1), y)
        calibrator = platt
        calib_kind = "platt"
    except Exception:
        calibrator = None
        calib_kind = "none"

def apply_calibrator(p):
    p = np.asarray(p, dtype=np.float32)
    if calibrator is None or calib_kind == "none":
        return p
    if calib_kind == "isotonic":
        return calibrator.transform(p).astype(np.float32)
    return calibrator.predict_proba(p.reshape(-1,1))[:,1].astype(np.float32)

oof_prob_cal = apply_calibrator(oof_prob)
if len(uniq_y) > 1:
    auc_c = roc_auc_score(y, oof_prob_cal)
    ap_c  = average_precision_score(y, oof_prob_cal)
else:
    auc_c = float("nan"); ap_c = float("nan")
print(f"OOF calibrated ({calib_kind}): AUC={auc_c:.5f} | AP={ap_c:.5f}")

# ----------------------------
# Threshold tuning using mask-score proxy (fold-mean objective)
# score = mean( pred_forg*Dice(pred,gt) + (1-pred_forg)*1[gt_empty] )
# Catatan: Dice(pred,gt) dihitung dari mask_pack vs gt union.
# ----------------------------
# Popcount LUT sudah ada: POPCNT

def unpackbits_to_mask(pack_u8: np.ndarray, h: int, w: int) -> np.ndarray:
    bits = np.unpackbits(pack_u8.reshape(-1).astype(np.uint8), axis=None)[: h*w]
    return bits.reshape(h, w).astype(bool)

def parse_mask_paths(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    if isinstance(val, (list, tuple)):
        return [str(x) for x in val if str(x)]
    if isinstance(val, np.ndarray):
        try:
            return [str(x) for x in val.reshape(-1).tolist() if str(x)]
        except Exception:
            return []
    if isinstance(val, str):
        s = val.strip()
        if s == "" or s.lower() in ("nan","none","null"):
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            for parser in ("json", "ast"):
                try:
                    out = json.loads(s) if parser == "json" else ast.literal_eval(s)
                    if isinstance(out, (list, tuple)):
                        return [str(x) for x in out if str(x)]
                except Exception:
                    pass
        return [s]
    return []

def to_2d_bool(arr, target_h: int, target_w: int):
    a = np.asarray(arr)
    a = np.squeeze(a)

    n_bits_target = int(target_h * target_w)
    n_bytes_target = (n_bits_target + 7) // 8

    if a.dtype == np.uint8:
        flat = a.reshape(-1)
        if flat.size == n_bytes_target:
            return unpackbits_to_mask(flat, target_h, target_w)

    if a.ndim == 3:
        if a.shape[0] == target_h and a.shape[1] == target_w:
            a = a.max(axis=2)
        elif a.shape[1] == target_h and a.shape[2] == target_w:
            a = a.max(axis=0)
        else:
            flat = a.reshape(-1)
            if flat.dtype == np.uint8 and flat.size == n_bytes_target:
                return unpackbits_to_mask(flat, target_h, target_w)

    if a.ndim == 1 and a.size == n_bits_target:
        a = a.reshape(target_h, target_w)

    if a.ndim != 2:
        return np.zeros((target_h, target_w), dtype=bool)

    m = (a > 0)
    if m.shape != (target_h, target_w):
        im = Image.fromarray((m.astype(np.uint8)*255)).resize((target_w, target_h), resample=Image.NEAREST)
        m = (np.array(im) > 0)
    return m

def load_mask_any_as_bool(path: str, target_h: int, target_w: int):
    p = Path(str(path))
    if not p.exists():
        return None
    suf = p.suffix.lower()

    if suf in (".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"):
        im = Image.open(p).convert("L")
        if im.size != (target_w, target_h):
            im = im.resize((target_w, target_h), resample=Image.NEAREST)
        return (np.array(im) > 0)

    if suf == ".npz":
        z = np.load(p, allow_pickle=False)
        if ("mask_pack" in z.files) and ("mask_h" in z.files) and ("mask_w" in z.files):
            mh = int(z["mask_h"]); mw = int(z["mask_w"])
            pack = z["mask_pack"].astype(np.uint8).reshape(-1)
            m0 = unpackbits_to_mask(pack, mh, mw)
            if (mh, mw) != (target_h, target_w):
                im = Image.fromarray((m0.astype(np.uint8)*255)).resize((target_w, target_h), resample=Image.NEAREST)
                return (np.array(im) > 0)
            return m0
        k0 = z.files[0] if len(z.files) else None
        return None if k0 is None else to_2d_bool(z[k0], target_h, target_w)

    try:
        arr = np.load(p, allow_pickle=False)
    except Exception:
        arr = np.load(p, allow_pickle=True)
        if np.ndim(arr) == 0 and hasattr(arr, "item"):
            arr = arr.item()
    return to_2d_bool(arr, target_h, target_w)

def pack_from_bool(mask_bool: np.ndarray) -> np.ndarray:
    return np.packbits(mask_bool.astype(np.uint8), axis=None)

def load_pred_pack(pred_path: str):
    z = np.load(pred_path, allow_pickle=False)
    pack = z["mask_pack"].astype(np.uint8).reshape(-1)
    h = int(z["mask_h"]); w = int(z["mask_w"])
    n_bits = int(h*w)
    return pack, n_bits, h, w

def load_gt_pack(mask_paths_val, target_h: int, target_w: int):
    n_bits = int(target_h * target_w)
    n_bytes = (n_bits + 7)//8
    paths = parse_mask_paths(mask_paths_val)
    if len(paths) == 0:
        return np.zeros((n_bytes,), dtype=np.uint8), n_bits

    union = np.zeros((target_h, target_w), dtype=bool)
    for p in paths:
        m = load_mask_any_as_bool(p, target_h, target_w)
        if m is None:
            continue
        union |= m
    return pack_from_bool(union).astype(np.uint8), n_bits

def dice_from_packs(pred_pack, gt_pack):
    pred_pack = pred_pack.reshape(-1).astype(np.uint8, copy=False)
    gt_pack   = gt_pack.reshape(-1).astype(np.uint8, copy=False)
    pred_area = int(POPCNT[pred_pack].sum()) if pred_pack.size else 0
    gt_area   = int(POPCNT[gt_pack].sum()) if gt_pack.size else 0

    if gt_area == 0 and pred_area == 0:
        return 1.0, pred_area, gt_area
    if gt_area == 0 and pred_area > 0:
        return 0.0, pred_area, gt_area
    if gt_area > 0 and pred_area == 0:
        return 0.0, pred_area, gt_area

    L = min(pred_pack.size, gt_pack.size)
    inter = int(POPCNT[np.bitwise_and(pred_pack[:L], gt_pack[:L])].sum())
    dice = float((2.0 * inter) / (pred_area + gt_area + 1e-12))
    return dice, pred_area, gt_area

print("\nPreloading pred/gt packs for threshold tuning ...")
t0 = time.time()

dice_pred = np.zeros(len(df), dtype=np.float32)
gt_empty  = np.zeros(len(df), dtype=np.float32)

for i, row in df.iterrows():
    ppack, n_bits, h, w = load_pred_pack(row["pred_path"])
    gtpack, _ = load_gt_pack(row["mask_paths"], h, w)

    d, pa, ga = dice_from_packs(ppack, gtpack)
    dice_pred[i] = np.float32(d)
    gt_empty[i]  = np.float32(1.0 if ga == 0 else 0.0)

dt = time.time() - t0
print(f"Preload done in {dt:.1f}s | gt_empty_rate={gt_empty.mean():.4f}")

# ----------------------------
# Threshold grid search (fold-mean score)
# ----------------------------
has_peak     = df["has_peak"].values.astype(np.float32)
area_frac    = df["pred_area_frac"].values.astype(np.float32)  # pakai yang dihitung ulang
inlier_ratio = df["inlier_ratio"].values.astype(np.float32)
oof = oof_prob_cal.astype(np.float32)

thr_p_list = np.linspace(0.05, 0.95, 37).astype(np.float32)

area_nz = area_frac[area_frac > 0]
if area_nz.size > 0:
    qs = np.unique(np.quantile(area_nz, [0.40,0.55,0.70,0.80,0.88,0.93,0.96,0.98,0.99]).astype(np.float32))
    thr_area_list = np.unique(np.concatenate([[0.0], qs])).astype(np.float32)
else:
    thr_area_list = np.array([0.0], dtype=np.float32)

thr_inlier_list = np.array([0.0, 0.10, 0.20, 0.28, 0.35, 0.42, 0.50], dtype=np.float32)

best = {"score": -1.0, "thr_forged": 0.5, "thr_area": 0.0, "thr_inlier": 0.0}

print("\nGrid search thresholds (objective=mean_dice_proxy_per_fold) ...")
t0 = time.time()

fold_ids = np.unique(folds)
for tp in thr_p_list:
    base = (oof >= tp) & (has_peak >= 0.5)
    for ta in thr_area_list:
        mid = base & (area_frac >= ta)
        for ti in thr_inlier_list:
            pred_forg = mid & (inlier_ratio >= ti)
            pred_f = pred_forg.astype(np.float32)

            # fold-mean score
            scores = []
            for f in fold_ids:
                m = (folds == f)
                if m.sum() == 0:
                    continue
                s = float(np.mean(pred_f[m] * dice_pred[m] + (1.0 - pred_f[m]) * gt_empty[m]))
                scores.append(s)
            mean_score = float(np.mean(scores)) if len(scores) else -1.0

            if mean_score > best["score"]:
                best.update({
                    "score": mean_score,
                    "thr_forged": float(tp),
                    "thr_area": float(ta),
                    "thr_inlier": float(ti),
                })

print(f"Threshold search done in {time.time()-t0:.1f}s")
print("\nBest thresholds (OOF, objective=mean_dice_proxy_per_fold):")
print(json.dumps(best, indent=2))

# ----------------------------
# Train final ensemble on ALL data & save artifacts
# ----------------------------
FINAL_DIR = Path("/kaggle/working/recodai_luc/models/gate_base_v3")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

lr_all = make_lr()
et_all = make_et()
hgb_all = make_hgb()

lr_all.fit(X, y)
et_all.fit(X, y)
hgb_all.fit(X, y, sample_weight=sample_weight)

final_model = {"lr": lr_all, "et": et_all, "hgb": hgb_all, "weights": [0.25, 0.40, 0.35]}

joblib.dump(final_model, FINAL_DIR / "gate_model.joblib")
joblib.dump(models_by_fold, FINAL_DIR / "gate_models_by_fold.joblib")
joblib.dump({"kind": calib_kind, "calibrator": calibrator}, FINAL_DIR / "calibrator.joblib")

with open(FINAL_DIR / "feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

thresholds = {
    "thr_forged": best["thr_forged"],  # dipakai STAGE 6
    "thr_p": best["thr_forged"],       # alias
    "thr_area": best["thr_area"],
    "thr_inlier": best["thr_inlier"],
    "require_has_peak": True,
}
with open(FINAL_DIR / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

# OOF audit
df_oof = df[["sample_id","case_id","variant","fold","y_forged"]].copy()
df_oof["p_cal"]  = oof_prob_cal.astype(np.float32)

pred_forg_best = ((oof_prob_cal >= thresholds["thr_forged"]) &
                  (df["has_peak"].values >= 0.5) &
                  (df["pred_area_frac"].values >= thresholds["thr_area"]) &
                  (df["inlier_ratio"].values >= thresholds["thr_inlier"])).astype(np.int8)
df_oof["pred_forg_best"] = pred_forg_best

pred_f = pred_forg_best.astype(np.float32)
best_proxy = float(np.mean(pred_f * dice_pred + (1.0 - pred_f) * gt_empty))

df_oof.to_csv(FINAL_DIR / "oof_predictions.csv", index=False)

report = {
    "n_train_with_pred": int(len(df)),
    "forged_rate": float(y.mean()),
    "n_folds": int(df["fold"].nunique()),
    "oof_auc_ensemble": float(auc) if np.isfinite(auc) else None,
    "oof_ap_ensemble": float(ap) if np.isfinite(ap) else None,
    "oof_auc_cal": float(auc_c) if np.isfinite(auc_c) else None,
    "oof_ap_cal": float(ap_c) if np.isfinite(ap_c) else None,
    "calibration": calib_kind,
    "thresholds": thresholds,
    "best_mean_dice_proxy": float(best_proxy),
    "feature_cols": FEATURE_COLS,
    "pred_manifest_used": str(PRED_MAN_TRAIN_PATH),
}
with open(FINAL_DIR / "report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nSAVED ARTIFACTS ->", FINAL_DIR)
print("Files:", sorted([p.name for p in FINAL_DIR.iterdir()]))

GATE_MODEL_DIR = str(FINAL_DIR)
print("\nDONE. Exported: GATE_MODEL_DIR")


PRED_MAN_TRAIN_PATH: /kaggle/working/recodai_luc/cache/pred_base_v2_v5/manifest_pred_train_all.csv
Train_all rows (with pred): 5,176 / total df_train_all: 5,176
Class rate forged: 0.5408 | folds: 5
Loaded pred scalars. Missing pred files during read: 0
OOF ensemble: AUC=1.00000 | AP=1.00000 | time_s=9.8
OOF calibrated (isotonic): AUC=1.00000 | AP=1.00000

Preloading pred/gt packs for threshold tuning ...
Preload done in 52.1s | gt_empty_rate=0.4592

Grid search thresholds (objective=mean_dice_proxy_per_fold) ...
Threshold search done in 0.1s

Best thresholds (OOF, objective=mean_dice_proxy_per_fold):
{
  "score": 0.4592350244522095,
  "thr_forged": 0.05000000074505806,
  "thr_area": 0.0,
  "thr_inlier": 0.0
}

SAVED ARTIFACTS -> /kaggle/working/recodai_luc/models/gate_base_v3
Files: ['calibrator.joblib', 'feature_cols.json', 'gate_model.joblib', 'gate_models_by_fold.joblib', 'oof_predictions.csv', 'report.json', 'thresholds.json']

DONE. Exported: GATE_MODEL_DIR


# Inference Strategy: Two-Pass + Smart Ensemble + Export RLE (Strict Guard)

In [6]:
# ============================================================
# STAGE 6 — Inference Strategy: Two-Pass + Smart Ensemble + Export RLE (Strict Guard)
# ONE CELL, REVISI FULL v2.1 (lebih cepat + lebih robust + kompatibel Gate v2 & v3).
#
# Upgrade utama:
# - Auto-detect PASS-1 pred cache terbaik (v5/v4/v2/v1) berdasarkan folder yang ada.
# - Gate inference VECTORIZED untuk semua test (cepat) + kompatibel:
#     * gate_model sklearn (predict_proba)
#     * gate_model dict ensemble {lr,et,hgb,weights}
# - Build gate features lebih lengkap (support feature_cols lama/baru):
#     area_frac dari mask_pack (popcount) -> lebih reliable daripada scalar
#     pred_area_frac / log_pred_area / sqrt_area / variant dummies (jika ada)
# - PASS-2 hanya untuk borderline/repair: selection lebih pintar + cap opsional.
# - On-the-fly robust_match memakai "filter neighbor setelah topk" (tanpa mask NxN).
# - Strict guard gunakan area_frac dari mask_pack + anti-fragment / anti-huge-mask.
# - RLE export strict sesuai sample_submission order.
# ============================================================

import os, gc, json, time, math
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from transformers import AutoModel, AutoImageProcessor
import joblib

# ----------------------------
# PATHS (fixed by user)
# ----------------------------
DATA_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
TEST_IMAGES_DIR = DATA_ROOT / "test_images"
SAMPLE_SUB_PATH = DATA_ROOT / "sample_submission.csv"
DINO_BASE_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")

# PASS-1 pred cache candidates (auto-pick best existing)
PRED1_CANDIDATES = [
    Path("/kaggle/working/recodai_luc/cache/pred_base_v2_v5/test"),
    Path("/kaggle/working/recodai_luc/cache/pred_base_v2_v4/test"),
    Path("/kaggle/working/recodai_luc/cache/pred_base_v2/test"),
    Path("/kaggle/working/recodai_luc/cache/pred_base/test"),
    Path("/kaggle/working/recodai_luc/cache/pred_base"),  # legacy layout
]

# PASS-2 cache (optional)
PRED2_DIR = Path("/kaggle/working/recodai_luc/cache/pred_base_p2_v2")
PRED2_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUB_PATH  = Path("/kaggle/working/submission.csv")
OUT_COPY_PATH = Path("/kaggle/working/recodai_luc/outputs/submission.csv")
OUT_COPY_PATH.parent.mkdir(parents=True, exist_ok=True)

# ----------------------------
# REQUIRE: sample submission
# ----------------------------
if not SAMPLE_SUB_PATH.exists():
    raise FileNotFoundError(f"sample_submission.csv not found: {SAMPLE_SUB_PATH}")

df_sample = pd.read_csv(SAMPLE_SUB_PATH)
if not {"case_id","annotation"}.issubset({c.lower() for c in df_sample.columns}):
    raise ValueError(f"sample_submission must contain case_id, annotation. Found: {list(df_sample.columns)}")

col_case = [c for c in df_sample.columns if c.lower()=="case_id"][0]
col_ann  = [c for c in df_sample.columns if c.lower()=="annotation"][0]
df_sample = df_sample.rename(columns={col_case:"case_id", col_ann:"annotation"}).copy()
df_sample["case_id"] = df_sample["case_id"].astype(str)

# ----------------------------
# Resolve df_test in sample order
# ----------------------------
IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}

def build_caseid_map(folder: Path) -> dict:
    mp = {}
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    for p in files:
        cid = p.stem
        if cid not in mp:
            mp[cid] = p
    return mp

test_img_map = build_caseid_map(TEST_IMAGES_DIR)

if "df_test" in globals() and isinstance(globals()["df_test"], pd.DataFrame) and "case_id" in globals()["df_test"].columns:
    df_test = globals()["df_test"].copy()
    if "image_path" not in df_test.columns:
        df_test["image_path"] = df_test["case_id"].astype(str).map(lambda x: str(test_img_map.get(str(x), "")))
else:
    df_test = pd.DataFrame({"case_id": sorted(test_img_map.keys())})
    df_test["image_path"] = df_test["case_id"].map(lambda x: str(test_img_map.get(str(x), "")))

df_test["case_id"] = df_test["case_id"].astype(str)
df_test["image_path"] = df_test["image_path"].astype(str)

# keep sample order strictly
df_test = df_sample[["case_id"]].merge(df_test, on="case_id", how="left")

n_ok = int(df_test["image_path"].map(lambda p: Path(p).exists()).sum())
print(f"Test images resolved: {n_ok:,}/{len(df_test):,}")

# ----------------------------
# RLE encode (use RLE_ORDER if exists, else 'F')
# ----------------------------
RLE_ORDER = globals().get("RLE_ORDER", "F")
if RLE_ORDER not in ("F","C"):
    RLE_ORDER = "F"

def rle_encode(mask: np.ndarray, order: str="F") -> str:
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return ""
    if order.upper() == "F":
        pixels = m.T.reshape(-1)
    else:
        pixels = m.reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# ----------------------------
# Load GATE artifacts (STAGE 5)
# - kompatibel:
#   * v2: gate_model sklearn pipeline
#   * v3: gate_model dict {"lr","et","hgb","weights"}
# ----------------------------
def _pick_existing_dir(*paths):
    for p in paths:
        p = Path(str(p))
        if p.exists():
            return p
    return None

if "GATE_MODEL_DIR" in globals():
    GATE_MODEL_DIR = Path(str(globals()["GATE_MODEL_DIR"]))
else:
    GATE_MODEL_DIR = _pick_existing_dir(
        "/kaggle/working/recodai_luc/models/gate_base_v3",
        "/kaggle/working/recodai_luc/models/gate_base_v2",
        "/kaggle/working/recodai_luc/models/gate_base_v1",
    )

if GATE_MODEL_DIR is None or (not GATE_MODEL_DIR.exists()):
    raise FileNotFoundError("GATE_MODEL_DIR not found (run STAGE 5).")

gate_model = joblib.load(GATE_MODEL_DIR / "gate_model.joblib")
calib_pack = joblib.load(GATE_MODEL_DIR / "calibrator.joblib")
calib_kind = calib_pack.get("kind","none")
calibrator = calib_pack.get("calibrator", None)

feature_cols = json.loads((GATE_MODEL_DIR / "feature_cols.json").read_text())
thresholds   = json.loads((GATE_MODEL_DIR / "thresholds.json").read_text())

thr_p = float(thresholds.get("thr_p", thresholds.get("thr_forged", 0.5)))
thr_area   = float(thresholds.get("thr_area", 0.0))
thr_inlier = float(thresholds.get("thr_inlier", 0.0))
require_has_peak = bool(thresholds.get("require_has_peak", True))

print("\nLoaded gate artifacts:")
print(f"  model_dir    : {GATE_MODEL_DIR}")
print(f"  calib_kind   : {calib_kind}")
print(f"  thr_p        : {thr_p}")
print(f"  thr_area     : {thr_area}")
print(f"  thr_inlier   : {thr_inlier}")
print(f"  require_peak : {require_has_peak}")
print(f"  RLE_ORDER    : {RLE_ORDER}")
print(f"  n_features   : {len(feature_cols)}")

def apply_calibrator(p):
    p = np.asarray(p, dtype=np.float32)
    if calibrator is None or calib_kind == "none":
        return p
    if calib_kind == "isotonic":
        return calibrator.transform(p).astype(np.float32)
    return calibrator.predict_proba(p.reshape(-1,1))[:,1].astype(np.float32)

def gate_predict_proba_matrix(X: np.ndarray) -> np.ndarray:
    """
    Return calibrated probability for class-1.
    Supports:
    - sklearn-like model with predict_proba
    - dict ensemble {lr,et,hgb,weights}
    """
    if isinstance(gate_model, dict) and all(k in gate_model for k in ["lr","et","hgb"]):
        w = gate_model.get("weights", [0.25, 0.40, 0.35])
        w = np.asarray(w, dtype=np.float32).reshape(-1)
        w = w / (w.sum() + 1e-12)
        p_lr  = gate_model["lr"].predict_proba(X)[:,1].astype(np.float32)
        p_et  = gate_model["et"].predict_proba(X)[:,1].astype(np.float32)
        p_hgb = gate_model["hgb"].predict_proba(X)[:,1].astype(np.float32)
        p = (w[0]*p_lr + w[1]*p_et + w[2]*p_hgb).astype(np.float32)
    else:
        p = gate_model.predict_proba(X)[:,1].astype(np.float32)
    return apply_calibrator(p)

# ----------------------------
# PASS-1 pred cache detection
# ----------------------------
PRED1_DIR = None
for cand in PRED1_CANDIDATES:
    if cand.exists():
        PRED1_DIR = cand
        break
if PRED1_DIR is None:
    # no cache -> create preferred folder to save pass1 fallback
    PRED1_DIR = Path("/kaggle/working/recodai_luc/cache/pred_base_v2/test")
    PRED1_DIR.mkdir(parents=True, exist_ok=True)

print("\nPred cache (PASS-1) dir:", PRED1_DIR)

# ----------------------------
# Pred NPZ loader (pack + scalars) — without unpacking mask by default
# ----------------------------
POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

SCALAR_KEYS = [
    "has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
    "inlier_ratio","pair_count","uniq_src","uniq_dst",
    "area_frac","n_comp","largest_comp",
    "n_pairs_thr","n_pairs_mnn",
]

def _get_scalar(z, k, default=0.0):
    if k not in z.files:
        return float(default)
    v = z[k]
    if np.ndim(v) == 0:
        return float(v)
    return float(np.array(v).reshape(-1)[0])

def load_pred_npz_pack(pred_dir: Path, case_id: str):
    p = pred_dir / f"{case_id}.npz"
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    mh = int(_get_scalar(z, "mask_h", 0))
    mw = int(_get_scalar(z, "mask_w", 0))
    pack = z["mask_pack"].astype(np.uint8, copy=False).reshape(-1) if "mask_pack" in z.files else np.zeros((0,), np.uint8)

    scal = {"case_id": case_id, "mask_h": float(mh), "mask_w": float(mw)}
    for k in SCALAR_KEYS:
        scal[k] = _get_scalar(z, k, 0.0)

    # recompute area from packbits (lebih reliable)
    denom = float(mh * mw) if (mh > 0 and mw > 0) else 0.0
    area = int(POPCNT[pack].sum()) if pack.size else 0
    area_frac = float(area / denom) if denom > 0 else 0.0
    scal["pred_area"] = float(area)
    scal["pred_area_frac"] = float(area_frac)

    # override scalar area_frac (biar konsisten dengan mask_pack)
    scal["area_frac"] = float(area_frac)

    return pack, mh, mw, scal

def unpack_mask_from_pack(pack: np.ndarray, h: int, w: int) -> np.ndarray:
    if h <= 0 or w <= 0 or pack is None or pack.size == 0:
        return np.zeros((max(h,1), max(w,1)), dtype=np.uint8)
    bits = np.unpackbits(pack.astype(np.uint8), axis=None)[: h*w]
    return bits.reshape(h, w).astype(np.uint8)

def save_pred_npz_from_mask(pred_dir: Path, case_id: str, mask: np.ndarray, scal: dict):
    pred_dir.mkdir(parents=True, exist_ok=True)
    mh, mw = mask.shape
    pack = np.packbits((mask > 0).astype(np.uint8), axis=None).astype(np.uint8)
    payload = {
        "mask_pack": pack,
        "mask_h": np.int32(mh),
        "mask_w": np.int32(mw),
    }
    for k in SCALAR_KEYS:
        if k in scal:
            payload[k] = np.float32(float(scal.get(k, 0.0)))
    if "p_gate" in scal:
        payload["p_gate"] = np.float32(float(scal["p_gate"]))
    np.savez_compressed(pred_dir / f"{case_id}.npz", **payload)

# ----------------------------
# Build gate feature vector (dynamic mengikuti feature_cols.json)
# - support v2 feature set + v3 feature set
# ----------------------------
def safe_log1p(x):
    x = np.asarray(x, dtype=np.float32)
    return np.log1p(np.clip(x, 0.0, None))

def build_gate_row(scal: dict, variant: str=""):
    mh = float(scal.get("mask_h", 0.0))
    mw = float(scal.get("mask_w", 0.0))
    denom = mh * mw if (mh > 0 and mw > 0) else 0.0

    peak_ratio = float(scal.get("peak_ratio", 0.0))
    best_count = float(scal.get("best_count", 0.0))
    pair_count = float(scal.get("pair_count", 0.0))
    n_comp = float(scal.get("n_comp", 0.0))
    largest_comp = float(scal.get("largest_comp", 0.0))

    pred_area = float(scal.get("pred_area", 0.0))
    pred_area_frac = float(scal.get("pred_area_frac", scal.get("area_frac", 0.0)))

    row = {
        # base
        "has_peak": float(scal.get("has_peak", 0.0)),
        "peak_ratio": peak_ratio,
        "best_weight": float(scal.get("best_weight", 0.0)),
        "best_mean_sim": float(scal.get("best_mean_sim", 0.0)),
        "inlier_ratio": float(scal.get("inlier_ratio", 0.0)),
        "area_frac": float(scal.get("area_frac", 0.0)),
        "pred_area": pred_area,
        "pred_area_frac": pred_area_frac,
        "uniq_src": float(scal.get("uniq_src", 0.0)),
        "uniq_dst": float(scal.get("uniq_dst", 0.0)),
        "pair_count": pair_count,
        "best_count": best_count,
        "n_comp": n_comp,
        "largest_comp": largest_comp,

        # derived common
        "peak_ratio_clip": float(np.clip(peak_ratio, 0.0, 80.0)),
        "largest_frac": float(largest_comp / denom) if denom > 0 else 0.0,
        "log_best_count": float(safe_log1p(best_count)),
        "log_pair_count": float(safe_log1p(pair_count)),
        "log_n_comp": float(safe_log1p(n_comp)),
        "log_largest": float(safe_log1p(largest_comp)),

        # derived v3
        "log_pred_area": float(safe_log1p(pred_area)),
        "sqrt_area": float(np.sqrt(np.clip(pred_area_frac, 0.0, 1.0))),
    }

    # optional extras
    n_thr = float(scal.get("n_pairs_thr", 0.0))
    n_mnn = float(scal.get("n_pairs_mnn", 0.0))
    row["log_thr_pairs"] = float(safe_log1p(n_thr))
    row["log_mnn_pairs"] = float(safe_log1p(n_mnn))

    # variant dummies: feature cols sering berbentuk v_xxx
    # kalau test tidak punya variant -> semuanya 0.
    if isinstance(variant, str) and variant != "":
        row[f"v_{variant}"] = 1.0

    return row

def build_gate_matrix(scals_list, variants_list):
    rows = []
    for scal, var in zip(scals_list, variants_list):
        rows.append(build_gate_row(scal, variant=var))
    # construct matrix in feature_cols order
    X = np.zeros((len(rows), len(feature_cols)), dtype=np.float32)
    for i, r in enumerate(rows):
        X[i, :] = np.array([r.get(c, 0.0) for c in feature_cols], dtype=np.float32)
    return X

# ----------------------------
# PASS-2 (on-the-fly compute) — lazy-load DINO
# - robust_match: filter neighbor AFTER topk (no NxN close mask)
# ----------------------------
torch.set_grad_enabled(False)

def get_device():
    return torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

_DINO = {"ready": False, "processor": None, "model": None, "device": None, "mean": None, "std": None}

def ensure_dino_loaded():
    if _DINO["ready"]:
        return
    if not DINO_BASE_DIR.exists():
        raise FileNotFoundError(f"DINO base not found: {DINO_BASE_DIR}")
    dev = get_device()
    processor = AutoImageProcessor.from_pretrained(str(DINO_BASE_DIR))
    model = AutoModel.from_pretrained(str(DINO_BASE_DIR)).eval().to(dev)
    mean = torch.tensor(processor.image_mean, dtype=torch.float32, device=dev).view(3,1,1)
    std  = torch.tensor(processor.image_std,  dtype=torch.float32, device=dev).view(3,1,1)
    _DINO.update({"ready": True, "processor": processor, "model": model, "device": dev, "mean": mean, "std": std})
    print(f"\nDINO loaded on: {dev}")

try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

from functools import lru_cache

@lru_cache(maxsize=256)
def _grid_rc_np(gh: int, gw: int):
    rr = np.repeat(np.arange(gh, dtype=np.int32), gw)
    cc = np.tile(np.arange(gw, dtype=np.int32), gh)
    return rr, cc

def resize_keep_aspect(orig_w, orig_h, max_side):
    if max(orig_w, orig_h) <= max_side:
        scale = 1.0
    else:
        scale = float(max_side) / float(max(orig_w, orig_h))
    new_w = max(1, int(round(orig_w * scale)))
    new_h = max(1, int(round(orig_h * scale)))
    return new_w, new_h

def align_to_patch(w, h, patch=14):
    w2 = max(patch, (w // patch) * patch)
    h2 = max(patch, (h // patch) * patch)
    return w2, h2

def preprocess_image(image_path: str, max_side: int, patch: int=14):
    ensure_dino_loaded()
    im = Image.open(image_path).convert("RGB")
    orig_w, orig_h = im.size
    new_w, new_h = resize_keep_aspect(orig_w, orig_h, max_side=max_side)
    new_w, new_h = align_to_patch(new_w, new_h, patch=patch)
    im2 = im.resize((new_w, new_h), resample=Image.BILINEAR) if (new_w, new_h) != (orig_w, orig_h) else im

    arr = (np.asarray(im2, dtype=np.float32) / 255.0)
    x = torch.from_numpy(arr).permute(2,0,1).contiguous().to(_DINO["device"])
    x = (x - _DINO["mean"]) / _DINO["std"]

    meta = {
        "orig_h": int(orig_h), "orig_w": int(orig_w),
        "proc_h": int(new_h),  "proc_w": int(new_w),
        "grid_h": int(new_h // patch),
        "grid_w": int(new_w // patch),
        "patch": int(patch),
    }
    return x, meta

def extract_dino_feat(image_path: str, max_side: int, patch: int=14):
    x, meta = preprocess_image(image_path, max_side=max_side, patch=patch)
    with torch.inference_mode():
        out = _DINO["model"](pixel_values=x.unsqueeze(0))
        hs = out.last_hidden_state  # [1,1+N,D]
        feat = hs[0, 1:, :].detach().to(torch.float32).cpu().numpy()
    return feat, meta

def canonicalize_offsets(dy: np.ndarray, dx: np.ndarray):
    dy = dy.astype(np.int32, copy=False)
    dx = dx.astype(np.int32, copy=False)
    neg = (dy < 0) | ((dy == 0) & (dx < 0))
    dy2 = dy.copy(); dx2 = dx.copy()
    dy2[neg] = -dy2[neg]; dx2[neg] = -dx2[neg]
    return dy2, dx2

def robust_match_from_feat_fast(feat_np: np.ndarray, gh: int, gw: int,
                                topk=60, sim_thr=0.75, min_sep=4,
                                bin_step=1, peaks_M=3, min_peak_count=20, weight_power=1.0):
    N = int(gh * gw)
    if feat_np is None or feat_np.ndim != 2 or feat_np.shape[0] != N:
        return {"has_peak": 0}

    f = torch.from_numpy(feat_np.astype(np.float32, copy=False))
    f = torch.nn.functional.normalize(f, dim=1)

    sim = f @ f.T
    sim.fill_diagonal_(-1e9)

    k = min(int(topk), N-1)
    vals, idxs = torch.topk(sim, k=k, dim=1, largest=True, sorted=False)

    vals_np = vals.cpu().numpy()
    idxs_np = idxs.cpu().numpy()

    src = np.repeat(np.arange(N, dtype=np.int32), k)
    dst = idxs_np.reshape(-1).astype(np.int32, copy=False)
    sv  = vals_np.reshape(-1).astype(np.float32, copy=False)

    # neighbor-close filter AFTER topk (tanpa close NxN)
    rr, cc = _grid_rc_np(gh, gw)
    dr = np.abs(rr[src] - rr[dst])
    dc = np.abs(cc[src] - cc[dst])
    close = (dr < int(min_sep)) & (dc < int(min_sep))
    if close.any():
        keep_far = ~close
        src = src[keep_far]; dst = dst[keep_far]; sv = sv[keep_far]

    keep = sv >= float(sim_thr)
    if keep.sum() == 0:
        return {"has_peak": 0}

    src = src[keep]; dst = dst[keep]; sv = sv[keep]
    n_pairs_thr = int(sv.size)

    # MNN
    codes_f = (src.astype(np.int64) * np.int64(N) + dst.astype(np.int64))
    codes_r = (dst.astype(np.int64) * np.int64(N) + src.astype(np.int64))
    mnn = np.isin(codes_f, codes_r, assume_unique=False)
    if mnn.sum() == 0:
        return {"has_peak": 0, "n_pairs_thr": n_pairs_thr, "n_pairs_mnn": 0}

    src = src[mnn]; dst = dst[mnn]; sv = sv[mnn]
    n_pairs_mnn = int(sv.size)

    src_r = src // gw; src_c = src % gw
    dst_r = dst // gw; dst_c = dst % gw
    dy = (dst_r - src_r).astype(np.int32)
    dx = (dst_c - src_c).astype(np.int32)

    dyc, dxc = canonicalize_offsets(dy, dx)

    if int(bin_step) > 1:
        dyb = (np.round(dyc / int(bin_step))).astype(np.int32) * int(bin_step)
        dxb = (np.round(dxc / int(bin_step))).astype(np.int32) * int(bin_step)
    else:
        dyb, dxb = dyc, dxc

    w = np.clip(sv, 0.0, 1.0)
    if float(weight_power) != 1.0:
        w = np.power(w, float(weight_power))

    keys = (dyb.astype(np.int64) << 32) ^ (dxb.astype(np.int64) & np.int64(0xffffffff))
    uniq, inv = np.unique(keys, return_inverse=True)
    sum_w = np.bincount(inv, weights=w, minlength=len(uniq)).astype(np.float32)
    cnt   = np.bincount(inv, minlength=len(uniq)).astype(np.int32)

    order = np.argsort(-sum_w)
    peaks = []
    for idx in order:
        if cnt[idx] < int(min_peak_count):
            continue
        key = int(uniq[idx])
        dy_pk = np.int32(key >> 32)
        dx_pk = np.int32(key & np.int64(0xffffffff))
        if dx_pk >= 2**31:
            dx_pk = dx_pk - 2**32
        peaks.append((int(dy_pk), int(dx_pk), float(sum_w[idx]), int(cnt[idx])))
        if len(peaks) >= int(peaks_M):
            break

    if len(peaks) == 0:
        return {"has_peak": 0, "n_pairs_thr": n_pairs_thr, "n_pairs_mnn": n_pairs_mnn}

    w1 = peaks[0][2]
    w2 = peaks[1][2] if len(peaks) > 1 else 0.0
    peak_ratio = float(w1 / (w2 + 1e-9)) if w2 > 0 else float(1e9)

    best_dy, best_dx, best_weight, best_count = peaks[0]
    same = (dyb == best_dy) & (dxb == best_dx)

    return {
        "has_peak": 1,
        "peak_ratio": float(peak_ratio),
        "best_weight": float(best_weight),
        "best_count": int(best_count),
        "best_mean_sim": float(np.mean(sv[same])) if same.any() else 0.0,
        "best_src": src[same].astype(np.int32, copy=False),
        "best_dst": dst[same].astype(np.int32, copy=False),
        "best_sim": sv[same].astype(np.float16, copy=False),
        "n_pairs_thr": n_pairs_thr,
        "n_pairs_mnn": n_pairs_mnn,
    }

def _binary_close(mask: np.ndarray, ks: int) -> np.ndarray:
    if not _HAS_SCIPY or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_closing(mask.astype(bool), structure=st).astype(np.uint8)

def _binary_open(mask: np.ndarray, ks: int) -> np.ndarray:
    if not _HAS_SCIPY or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_opening(mask.astype(bool), structure=st).astype(np.uint8)

def _filter_components(mask: np.ndarray, min_area: int, keep_topk: int):
    if mask.sum() == 0:
        return mask.astype(np.uint8), 0, 0
    if not _HAS_SCIPY:
        area = int(mask.sum())
        if area < min_area:
            return np.zeros_like(mask, dtype=np.uint8), 0, 0
        return mask.astype(np.uint8), 1, area
    lab, n = ndi.label(mask.astype(bool))
    if n == 0:
        return mask.astype(np.uint8), 0, 0
    areas = np.bincount(lab.ravel())
    areas[0] = 0
    comps = np.where(areas >= int(min_area))[0]
    if comps.size == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0
    comps = comps[np.argsort(areas[comps])[::-1]]
    if keep_topk and int(keep_topk) > 0:
        comps = comps[:int(keep_topk)]
    out = np.isin(lab, comps).astype(np.uint8)
    largest = int(areas[comps[0]]) if comps.size else 0
    return out, int(comps.size), largest

def reconstruct_mask(meta: dict, match: dict,
                     sim_inlier_thr=0.80, thr_grid=0.25, score_mix=0.7,
                     grid_dilate=1, close_ks=3, open_ks=0,
                     min_area_frac=0.0003, keep_topk_comp=10,
                     thr_grid_min=0.12, thr_grid_max=0.40):
    gh, gw = int(meta["grid_h"]), int(meta["grid_w"])
    patch = int(meta["patch"])
    proc_h, proc_w = int(meta["proc_h"]), int(meta["proc_w"])
    orig_h, orig_w = int(meta["orig_h"]), int(meta["orig_w"])

    if not match or int(match.get("has_peak",0)) != 1:
        mask0 = np.zeros((orig_h, orig_w), dtype=np.uint8)
        scal = {k: 0.0 for k in SCALAR_KEYS}
        scal.update({"mask_h": float(orig_h), "mask_w": float(orig_w), "pred_area": 0.0, "pred_area_frac": 0.0})
        return mask0, scal

    src = match.get("best_src", np.zeros((0,), dtype=np.int32))
    dst = match.get("best_dst", np.zeros((0,), dtype=np.int32))
    sim = match.get("best_sim", np.zeros((0,), dtype=np.float16)).astype(np.float32, copy=False)

    if src.size == 0:
        mask0 = np.zeros((orig_h, orig_w), dtype=np.uint8)
        scal = {k: 0.0 for k in SCALAR_KEYS}
        scal.update({"mask_h": float(orig_h), "mask_w": float(orig_w), "pred_area": 0.0, "pred_area_frac": 0.0})
        return mask0, scal

    inlier_ratio = float((sim >= float(sim_inlier_thr)).mean())

    # dynamic thr_grid (smart): kualitas tinggi -> sedikit lebih longgar, kualitas rendah -> lebih ketat
    pr = float(match.get("peak_ratio", 0.0))
    bc = float(match.get("best_count", 0.0))
    t = float(thr_grid)
    if pr >= 4.0 and inlier_ratio >= 0.35:
        t -= 0.03
    if pr <= 1.5 or inlier_ratio <= 0.18:
        t += 0.04
    if bc >= 60:
        t -= 0.02
    t = float(np.clip(t, thr_grid_min, thr_grid_max))

    N = gh * gw
    score = np.zeros((N,), dtype=np.float32)
    count = np.zeros((N,), dtype=np.int32)
    w = np.clip(sim, 0.0, 1.0)

    np.add.at(score, src, w); np.add.at(score, dst, w)
    np.add.at(count, src, 1); np.add.at(count, dst, 1)

    smax = float(score.max()) if score.size else 0.0
    cmax = float(count.max()) if count.size else 0.0
    score_norm = score / (smax + 1e-9) if smax > 0 else score
    count_norm = count.astype(np.float32) / (cmax + 1e-9) if cmax > 0 else count.astype(np.float32)

    comb = float(score_mix) * score_norm + (1.0 - float(score_mix)) * count_norm
    grid = comb.reshape(gh, gw).astype(np.float32)

    mask_grid = (grid >= float(t)).astype(np.uint8)

    # grid dilate
    if int(grid_dilate) > 0:
        if _HAS_SCIPY:
            st = np.ones((3,3), dtype=bool)
            mg = mask_grid.astype(bool)
            for _ in range(int(grid_dilate)):
                mg = ndi.binary_dilation(mg, structure=st)
            mask_grid = mg.astype(np.uint8)
        else:
            for _ in range(int(grid_dilate)):
                pad = np.pad(mask_grid, 1, mode="constant")
                out = np.zeros_like(mask_grid)
                for dy in (-1,0,1):
                    for dx in (-1,0,1):
                        out = np.maximum(out, pad[1+dy:1+dy+mask_grid.shape[0], 1+dx:1+dx+mask_grid.shape[1]])
                mask_grid = out

    # upsample to proc then to orig
    mask_proc = np.kron(mask_grid.astype(np.uint8), np.ones((patch, patch), dtype=np.uint8))
    mask_proc = mask_proc[:proc_h, :proc_w]
    if (proc_h, proc_w) != (orig_h, orig_w):
        im = Image.fromarray((mask_proc * 255).astype(np.uint8))
        im = im.resize((orig_w, orig_h), resample=Image.NEAREST)
        mask_orig = (np.array(im) > 0).astype(np.uint8)
    else:
        mask_orig = mask_proc.astype(np.uint8)

    # morphology
    if int(close_ks) and int(close_ks) > 1:
        mask_orig = _binary_close(mask_orig, int(close_ks))
    if int(open_ks) and int(open_ks) > 1:
        mask_orig = _binary_open(mask_orig, int(open_ks))

    # components filter
    min_area = int(float(min_area_frac) * float(orig_h * orig_w))
    min_area = max(1, min_area)
    mask_orig, n_comp, largest = _filter_components(mask_orig, min_area=min_area, keep_topk=int(keep_topk_comp))

    area = int(mask_orig.sum())
    denom = float(orig_h * orig_w) + 1e-9
    area_frac = float(area / denom)

    scal = {
        "has_peak": float(match.get("has_peak", 0.0)),
        "peak_ratio": float(match.get("peak_ratio", 0.0)),
        "best_weight": float(match.get("best_weight", 0.0)),
        "best_count": float(match.get("best_count", 0.0)),
        "best_mean_sim": float(match.get("best_mean_sim", 0.0)),
        "inlier_ratio": float(inlier_ratio),
        "pair_count": float(sim.size),
        "uniq_src": float(np.unique(src).size),
        "uniq_dst": float(np.unique(dst).size),
        "area_frac": float(area_frac),
        "n_comp": float(n_comp),
        "largest_comp": float(largest),
        "mask_h": float(orig_h),
        "mask_w": float(orig_w),
        "n_pairs_thr": float(match.get("n_pairs_thr", 0.0)),
        "n_pairs_mnn": float(match.get("n_pairs_mnn", 0.0)),
        "pred_area": float(area),
        "pred_area_frac": float(area_frac),
        # debug / trace
        "thr_grid_used": float(t),
    }
    return mask_orig, scal

def run_on_the_fly(image_path: str, max_side: int, match_cfg: dict, recon_cfg: dict, variant: str=""):
    feat, meta = extract_dino_feat(image_path, max_side=max_side, patch=14)
    mch = robust_match_from_feat_fast(feat, meta["grid_h"], meta["grid_w"], **match_cfg)
    mask, scal = reconstruct_mask(meta, mch, **recon_cfg)
    # gate prob
    X1 = build_gate_matrix([scal], [variant])
    p = gate_predict_proba_matrix(X1)[0]
    scal["p_gate"] = float(p)
    return mask, scal

# ----------------------------
# PASS configs
# ----------------------------
PASS1_ONFLY = {
    "max_side": 448,
    "match_cfg": dict(topk=60, sim_thr=0.75, min_sep=4, bin_step=1, peaks_M=3, min_peak_count=20, weight_power=1.0),
    "recon_cfg": dict(sim_inlier_thr=0.80, thr_grid=0.25, score_mix=0.7, grid_dilate=1,
                      close_ks=3, open_ks=0,
                      min_area_frac=max(thr_area, 0.00025), keep_topk_comp=10),
}
PASS2 = {
    "max_side": 704,  # sedikit lebih detail dari 672
    "match_cfg": dict(topk=72, sim_thr=0.74, min_sep=4, bin_step=1, peaks_M=3, min_peak_count=22, weight_power=1.0),
    "recon_cfg": dict(sim_inlier_thr=0.80, thr_grid=0.24, score_mix=0.7, grid_dilate=1,
                      close_ks=3, open_ks=0,
                      min_area_frac=max(thr_area, 0.00025), keep_topk_comp=10),
}

# Borderline policy
BORDER_MARGIN = 0.10
MAX_BORDERLINE = None  # set int kalau mau batasi (mis. 1500)

def quality_score(scal: dict):
    p = float(scal.get("p_gate", 0.0))
    inl = float(scal.get("inlier_ratio", 0.0))
    pr = float(np.clip(float(scal.get("peak_ratio", 0.0)), 0.0, 10.0)) / 10.0
    area = float(scal.get("area_frac", scal.get("pred_area_frac", 0.0)))
    ncomp = float(scal.get("n_comp", 0.0))

    # penalties
    frag_pen = min(ncomp / 25.0, 1.0) * 0.45
    huge_pen = 0.0
    if area > 0.30:
        huge_pen = min((area - 0.30) / 0.30, 1.0) * 0.40

    q = p * (0.55 + 0.45*inl) * (0.25 + 0.75*pr) * (1.0 - frag_pen) * (1.0 - huge_pen)
    return float(q)

def strict_guard(scal: dict, pred_area: float):
    if require_has_peak and float(scal.get("has_peak", 0.0)) < 0.5:
        return False
    if float(scal.get("p_gate", 0.0)) < thr_p:
        return False

    area_frac = float(scal.get("area_frac", scal.get("pred_area_frac", 0.0)))
    if area_frac < thr_area:
        return False
    if float(scal.get("inlier_ratio", 0.0)) < thr_inlier:
        return False

    # extra anti-FP heuristics
    n_comp = float(scal.get("n_comp", 0.0))
    if n_comp > 45 and area_frac < max(thr_area, 0.0012):
        return False
    if area_frac > 0.65 and float(scal.get("p_gate",0.0)) < (thr_p + 0.15):
        return False
    if pred_area <= 0:
        return False
    return True

# ----------------------------
# MAIN
# ----------------------------
t0 = time.time()

# Optional: variant in test (kalau ada), else empty string
if "variant" in df_test.columns:
    test_variant = df_test["variant"].fillna("").astype(str).tolist()
else:
    test_variant = [""] * len(df_test)

# PASS-1: load cache packs + scalars; if missing -> on-the-fly pass1 per-case
packs1 = [None] * len(df_test)
hw1    = [None] * len(df_test)
scals1 = [None] * len(df_test)
cache_missing = 0
dino_needed = False

for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    ip  = str(row["image_path"])
    var = test_variant[i]

    loaded = load_pred_npz_pack(PRED1_DIR, cid)
    if loaded is None:
        # fallback compute PASS1 if image exists
        if Path(ip).exists():
            dino_needed = True
            try:
                mask1, scal = run_on_the_fly(ip, max_side=int(PASS1_ONFLY["max_side"]),
                                             match_cfg=PASS1_ONFLY["match_cfg"],
                                             recon_cfg=PASS1_ONFLY["recon_cfg"],
                                             variant=var)
                # save to PASS1 cache
                save_pred_npz_from_mask(PRED1_DIR, cid, mask1, scal)
                pack = np.packbits((mask1 > 0).astype(np.uint8), axis=None).astype(np.uint8)
                mh, mw = mask1.shape
                denom = float(mh*mw) if (mh>0 and mw>0) else 0.0
                area = int(mask1.sum())
                scal["pred_area"] = float(area)
                scal["pred_area_frac"] = float(area/denom) if denom>0 else 0.0
                scal["area_frac"] = float(scal["pred_area_frac"])
                packs1[i] = pack
                hw1[i] = (mh, mw)
                scals1[i] = scal
                cache_missing += 1
            except Exception:
                packs1[i] = np.zeros((0,), dtype=np.uint8)
                hw1[i] = (1,1)
                scals1[i] = {k: 0.0 for k in SCALAR_KEYS}
                scals1[i].update({"mask_h": 1.0, "mask_w": 1.0, "pred_area": 0.0, "pred_area_frac": 0.0, "area_frac": 0.0})
        else:
            packs1[i] = np.zeros((0,), dtype=np.uint8)
            hw1[i] = (1,1)
            scals1[i] = {k: 0.0 for k in SCALAR_KEYS}
            scals1[i].update({"mask_h": 1.0, "mask_w": 1.0, "pred_area": 0.0, "pred_area_frac": 0.0, "area_frac": 0.0})
    else:
        pack, mh, mw, scal = loaded
        packs1[i] = pack
        hw1[i] = (mh, mw)
        scals1[i] = scal

print(f"\nPASS-1 cache loaded. computed_onfly={cache_missing:,} | dino_needed={bool(dino_needed)}")

# Vectorized gate probability for PASS-1
X1 = build_gate_matrix(scals1, test_variant)
p1 = gate_predict_proba_matrix(X1).astype(np.float32)
for i in range(len(scals1)):
    scals1[i]["p_gate"] = float(p1[i])

# Borderline selection
borderline_ids = []
for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    p = float(scals1[i].get("p_gate", 0.0))
    area = float(scals1[i].get("area_frac", 0.0))
    inl = float(scals1[i].get("inlier_ratio", 0.0))
    hp = float(scals1[i].get("has_peak", 0.0))

    # conditions:
    near_thr = (abs(p - thr_p) <= BORDER_MARGIN) and (hp >= 0.5)
    # suspicious: predicted forged-ish but guard fails due to area/inlier
    repair = (p >= (thr_p - 0.06)) and (hp >= 0.5) and ((area < max(thr_area, 0.0009)) or (inl < thr_inlier + 0.06))
    # potential FN: p slightly below thr but area/inlier decent
    recover = (p >= (thr_p - 0.10)) and (hp >= 0.5) and (inl >= max(thr_inlier - 0.04, 0.0)) and (area >= max(thr_area*0.7, 0.00025))

    if (near_thr or repair or recover) and Path(str(row["image_path"])).exists():
        borderline_ids.append((cid, abs(p - thr_p)))

# cap optional
borderline_ids.sort(key=lambda x: x[1])
if MAX_BORDERLINE is not None and len(borderline_ids) > int(MAX_BORDERLINE):
    borderline_ids = borderline_ids[:int(MAX_BORDERLINE)]
borderline_caseids = [x[0] for x in borderline_ids]

print(f"Borderline for PASS-2: {len(borderline_caseids):,}/{len(df_test):,}")

# PASS-2 compute (only borderline)
p2_cache = {}  # cid -> (mask, scal)
if len(borderline_caseids) > 0:
    # load DINO once if needed
    try:
        ensure_dino_loaded()
    except Exception as e:
        print(f"[WARN] DINO not available for PASS-2: {repr(e)}")
        borderline_caseids = []

for j, cid in enumerate(borderline_caseids, 1):
    idx = int(np.where(df_test["case_id"].values.astype(str) == str(cid))[0][0])
    ip = str(df_test.iloc[idx]["image_path"])
    var = test_variant[idx]
    if not Path(ip).exists():
        continue
    try:
        mask2, scal2 = run_on_the_fly(ip, max_side=int(PASS2["max_side"]),
                                      match_cfg=PASS2["match_cfg"],
                                      recon_cfg=PASS2["recon_cfg"],
                                      variant=var)
        p2_cache[cid] = (mask2, scal2)
        # save pass2
        try:
            save_pred_npz_from_mask(PRED2_DIR, cid, mask2, scal2)
        except Exception:
            pass

        if (j % 100) == 0:
            print(f"[PASS2] {j:,}/{len(borderline_caseids):,} done | last={cid}")

    except Exception as e:
        if j <= 10:
            print(f"[WARN] PASS2 failed case_id={cid} err={repr(e)}")
        continue

    if (j % 200) == 0:
        gc.collect()

# Ensemble + Strict Guard + Export
results = []
n_pred_forg = 0
n_used_p2 = 0
n_union = 0
n_inter = 0

for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    pack1 = packs1[i]
    mh1, mw1 = hw1[i]
    scal1 = scals1[i]
    var = test_variant[i]

    # pass1 area (from pack)
    area1 = float(scal1.get("pred_area", 0.0))
    guard1 = strict_guard(scal1, pred_area=area1)
    q1 = quality_score(scal1)

    final_source = "p1"
    final_mask = None  # only build when needed
    final_scal = scal1
    final_area = area1
    final_guard = guard1
    final_q = q1

    if cid in p2_cache:
        mask2, scal2 = p2_cache[cid]
        # make sure p2 has p_gate (already inside scal2)
        area2 = float(mask2.sum())
        denom2 = float(mask2.size) + 1e-9
        scal2["pred_area"] = float(area2)
        scal2["pred_area_frac"] = float(area2 / denom2)
        scal2["area_frac"] = float(scal2["pred_area_frac"])
        guard2 = strict_guard(scal2, pred_area=area2)
        q2 = quality_score(scal2)

        # decision rules:
        # 1) if one guard true and other false -> take guarded
        # 2) if both guard true:
        #    - if IoU high -> union (recall up)
        #    - if IoU very low but both strong -> intersection (precision up)
        #    - else take higher quality
        # 3) if both guard false -> take higher quality (but likely authentic anyway)
        if guard2 and (not guard1):
            final_source = "p2"; final_mask = mask2; final_scal = scal2; final_area = area2; final_guard = guard2; final_q = q2
        elif guard1 and (not guard2):
            final_source = "p1"
        else:
            # both same guard state
            if guard1 and guard2:
                # need mask1 for iou
                mask1 = unpack_mask_from_pack(pack1, mh1, mw1)
                inter = float(np.logical_and(mask1 > 0, mask2 > 0).sum())
                uni   = float(np.logical_or(mask1 > 0, mask2 > 0).sum()) + 1e-9
                iou   = inter / uni

                strong1 = (float(scal1.get("p_gate",0.0)) >= thr_p + 0.12) and (float(scal1.get("inlier_ratio",0.0)) >= thr_inlier + 0.05)
                strong2 = (float(scal2.get("p_gate",0.0)) >= thr_p + 0.12) and (float(scal2.get("inlier_ratio",0.0)) >= thr_inlier + 0.05)

                if iou >= 0.18 and strong1 and strong2:
                    # union
                    mask_u = np.logical_or(mask1 > 0, mask2 > 0).astype(np.uint8)
                    area_u = float(mask_u.sum())
                    scal_u = dict(scal2)
                    scal_u["pred_area"] = area_u
                    scal_u["pred_area_frac"] = float(area_u / (mask_u.size + 1e-9))
                    scal_u["area_frac"] = float(scal_u["pred_area_frac"])
                    final_source = "union"; final_mask = mask_u; final_scal = scal_u; final_area = area_u; final_guard = True; final_q = max(q1, q2)
                    n_union += 1
                elif iou <= 0.06 and strong1 and strong2:
                    # intersection (anti-FP)
                    mask_i = np.logical_and(mask1 > 0, mask2 > 0).astype(np.uint8)
                    area_i = float(mask_i.sum())
                    scal_i = dict(scal2)
                    scal_i["pred_area"] = area_i
                    scal_i["pred_area_frac"] = float(area_i / (mask_i.size + 1e-9))
                    scal_i["area_frac"] = float(scal_i["pred_area_frac"])
                    # re-check guard for intersection (area might shrink)
                    if strict_guard(scal_i, pred_area=area_i):
                        final_source = "inter"; final_mask = mask_i; final_scal = scal_i; final_area = area_i; final_guard = True; final_q = max(q1, q2)
                        n_inter += 1
                    else:
                        # fallback pick best quality
                        if q2 > q1:
                            final_source = "p2"; final_mask = mask2; final_scal = scal2; final_area = area2; final_guard = guard2; final_q = q2
                        else:
                            final_source = "p1"
                else:
                    # pick best quality
                    if q2 > q1 * 1.03:
                        final_source = "p2"; final_mask = mask2; final_scal = scal2; final_area = area2; final_guard = guard2; final_q = q2
                    else:
                        final_source = "p1"
            else:
                # both guard false
                if q2 > q1 * 1.10:
                    final_source = "p2"; final_mask = mask2; final_scal = scal2; final_area = area2; final_guard = guard2; final_q = q2
                else:
                    final_source = "p1"

        if final_source in ("p2","union","inter"):
            n_used_p2 += 1

    # final annotation
    if final_guard:
        if final_mask is None:
            # need build mask from pass1 pack for RLE
            final_mask = unpack_mask_from_pack(pack1, mh1, mw1)
        rle = rle_encode(final_mask.astype(np.uint8), order=RLE_ORDER)
        ann = rle if rle != "" else "authentic"
        if ann != "authentic":
            n_pred_forg += 1
    else:
        ann = "authentic"

    results.append({"case_id": cid, "annotation": ann})

df_sub = pd.DataFrame(results)
df_sub = df_sample[["case_id"]].merge(df_sub, on="case_id", how="left")
df_sub["annotation"] = df_sub["annotation"].fillna("authentic").astype(str)

df_sub.to_csv(OUT_SUB_PATH, index=False)
df_sub.to_csv(OUT_COPY_PATH, index=False)

dt = time.time() - t0
print("\nDONE.")
print(f"submission.csv -> {OUT_SUB_PATH}")
print(f"copy          -> {OUT_COPY_PATH}")
print(f"elapsed_s     -> {dt:.1f}")
print(f"pred_forged   -> {n_pred_forg:,}/{len(df_sub):,}")
print(f"used_pass2    -> {n_used_p2:,} | union={n_union:,} | inter={n_inter:,}")
print(df_sub.head())


Test images resolved: 1/1

Loaded gate artifacts:
  model_dir    : /kaggle/working/recodai_luc/models/gate_base_v3
  calib_kind   : isotonic
  thr_p        : 0.05000000074505806
  thr_area     : 0.0
  thr_inlier   : 0.0
  require_peak : True
  RLE_ORDER    : F
  n_features   : 25

Pred cache (PASS-1) dir: /kaggle/working/recodai_luc/cache/pred_base_v2_v5/test

PASS-1 cache loaded. computed_onfly=0 | dino_needed=False


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Borderline for PASS-2: 1/1

DINO loaded on: cpu

DONE.
submission.csv -> /kaggle/working/submission.csv
copy          -> /kaggle/working/recodai_luc/outputs/submission.csv
elapsed_s     -> 3.9
pred_forged   -> 1/1
used_pass2    -> 1 | union=0 | inter=0
  case_id                                         annotation
0      45  1485 120 2648 161 2929 120 4092 161 4373 120 5...
